In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/ziyadaltalhi/tik-tok-data/Tik Tok Datasets - After Processing/المنطقة الوسطى/Tik Data/القصيم/dataset_tiktok-comments-scraper_2025-11-05_23-34-09-183_textready_analysis.xlsx
/kaggle/input/datasets/ziyadaltalhi/tik-tok-data/Tik Tok Datasets - After Processing/المنطقة الوسطى/Tik Data/القصيم/dataset_tiktok-comments-scraper_2025-11-05_23-06-43-625_textready_analysis.xlsx
/kaggle/input/datasets/ziyadaltalhi/tik-tok-data/Tik Tok Datasets - After Processing/المنطقة الوسطى/Tik Data/القصيم/dataset_tiktok-comments-scraper_2025-11-05_23-17-06-320_textready_cleaned.xlsx
/kaggle/input/datasets/ziyadaltalhi/tik-tok-data/Tik Tok Datasets - After Processing/المنطقة الوسطى/Tik Data/القصيم/dataset_tiktok-comments-scraper_2025-11-05_23-37-52-643_textready_analysis.xlsx
/kaggle/input/datasets/ziyadaltalhi/tik-tok-data/Tik Tok Datasets - After Processing/المنطقة الوسطى/Tik Data/القصيم/dataset_tiktok-comments-scraper_2025-11-05_23-36-16-596_textready_analysis.xlsx
/kaggle/input/dataset

In [2]:
import os

root_folder = r"/kaggle/input/datasets/ziyadaltalhi/tik-tok-data/Tik Tok Datasets - After Processing"

print("Exists:", os.path.exists(root_folder))
print("Is dir:", os.path.isdir(root_folder))
print("First level contents:")
print(os.listdir(root_folder)[:20] if os.path.exists(root_folder) else "Path not found")

Exists: True
Is dir: True
First level contents:
['المنطقة الوسطى', 'المنطقة الجنوبية', 'المنطقة الشمالية', 'المنطقة الشرقية', 'المنطقة الغربية']


In [3]:
# =========================================================
# TikTok LSTM Sentiment Model
# Same philosophy as Google Maps model
# But rebuilt cleanly for TikTok *_analysis.xlsx files only
# =========================================================

# =========================================================
# STEP 0: Imports
# =========================================================
import os
import re
import json
import pickle
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.utils import resample

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SpatialDropout1D, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

warnings.filterwarnings("ignore")

# =========================================================
# STEP 1: Settings
# =========================================================
ROOT_FOLDER = r"/kaggle/input/datasets/ziyadaltalhi/tik-tok-data/Tik Tok Datasets - After Processing"  
# Example:
# ROOT_FOLDER = r"/kaggle/input/tiktok-analysis/Tik Tok Datasets - After Processing"

OUTPUT_DIR = r"/kaggle/working/tiktok_lstm_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------
# Label mode options:
# 1) "sentiment_multiclass" -> uses Sentiment (negative/neutral/positive)
# 2) "stars_binary"         -> uses Stars (1,2=negative ; 4,5=positive ; 3 removed)
# ------------------------------
LABEL_MODE = "sentiment_multiclass"

# Choose preferred text column
TEXT_COLUMN_CANDIDATES = [
    "Text_TR",
    "Text_ML",
    "Text_Cleaned",
    "Text_Normalized",
    "Text",
    "Text_Orig",
    "text"
]

# Optional data quality filters
USE_ONLY_USEFUL_COMMENTS = False
MIN_QUALITY_SCORE = None        # Example: 6
MIN_COMMENT_STRENGTH = None     # Example: 5

# Text processing
APPLY_EXTRA_CLEANING = True
MIN_WORDS = 2

# Training settings
TEST_SIZE = 0.10
VAL_SIZE = 0.10   # from the remaining training set
VOCAB_SIZE = 50000
EMBEDDING_DIM = 128
LSTM_UNITS = 128
DENSE_UNITS = 64
DROPOUT_RATE = 0.30
BATCH_SIZE = 256
EPOCHS = 15
RANDOM_STATE = 42

# Sequence length
MAX_LEN_QUANTILE = 0.95
MIN_MAX_LEN = 40
MAX_MAX_LEN = 250

# Balancing
USE_OVERSAMPLING = False
USE_CLASS_WEIGHTS = True

# Label noise check
RUN_SANITY_CHECK = True
DROP_CONTRADICTORY_LABELS = False

# Aspect extraction
RUN_ASPECT_ANALYSIS = True

# Save model name
MODEL_NAME_PREFIX = f"tiktok_lstm_{LABEL_MODE}"

# =========================================================
# STEP 2: Reproducibility
# =========================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(RANDOM_STATE)

# =========================================================
# STEP 3: Arabic / general text cleaning
# =========================================================
AR_DIACRITICS = re.compile(r'[\u0617-\u061A\u064B-\u0652]')
URL_RE = re.compile(r'https?://\S+|www\.\S+')
EMAIL_RE = re.compile(r'\S+@\S+')
MENTION_RE = re.compile(r'@\w+')
HASHTAG_RE = re.compile(r'#(\w+)')
MULTISPACE_RE = re.compile(r'\s+')
REPEAT_CHARS_RE = re.compile(r'(.)\1{2,}')

def normalize_arabic(text: str) -> str:
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"ئ", "ي", text)
    text = re.sub(r"ة", "ه", text)
    text = re.sub(r"ـ", "", text)
    text = AR_DIACRITICS.sub("", text)
    return text

def clean_text(text: str) -> str:
    if pd.isna(text):
        return ""

    text = str(text).strip().lower()

    if text in {"", "nan", "none", "null", "[]", "{}"}:
        return ""

    text = URL_RE.sub(" ", text)
    text = EMAIL_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    text = HASHTAG_RE.sub(r" \1 ", text)

    text = normalize_arabic(text)

    # Keep Arabic, English, digits, spaces
    text = re.sub(r"[^0-9a-zA-Z\u0600-\u06FF\s]", " ", text)

    # Reduce repeated letters
    text = REPEAT_CHARS_RE.sub(r"\1\1", text)

    text = MULTISPACE_RE.sub(" ", text).strip()
    return text

def word_count(text: str) -> int:
    if not isinstance(text, str) or not text.strip():
        return 0
    return len(text.split())

# =========================================================
# STEP 4: Aspect keywords
# =========================================================
ASPECTS = {
    "المنظر_والطبيعه": [
        "منظر", "اطلاله", "اطلالة", "طبيعه", "طبيعة", "جميل", "روعه", "روعة"
    ],
    "الاسعار": [
        "سعر", "اسعار", "غالي", "رخيص", "مبالغ", "مبالغه", "مبالغة"
    ],
    "النظافه": [
        "نظيف", "نظافه", "نظافة", "وسخ", "قذر", "متسخ"
    ],
    "الخدمه": [
        "خدمه", "خدمة", "تعامل", "موظف", "موظفين", "استقبال", "خدمات"
    ],
    "المواقف": [
        "موقف", "مواقف", "سيارات", "باركنج", "parking"
    ],
    "الازدحام": [
        "زحمه", "زحمة", "ازدحام", "مزدحم", "هدوء", "زحام"
    ],
    "الاجواء": [
        "اجواء", "أجواء", "جو", "الجو", "بارد", "حار", "حر", "معتدل"
    ],
    "الموقع": [
        "موقع", "الموقع", "مكان", "بعيد", "قريب", "سهل", "صعب"
    ]
}

def extract_aspects(text: str, aspects_dict: dict) -> list:
    found = []
    if not isinstance(text, str) or not text.strip():
        return found

    for aspect_name, keywords in aspects_dict.items():
        for kw in keywords:
            if re.search(rf"\b{re.escape(kw)}\b", text):
                found.append(aspect_name)
                break
    return found

# =========================================================
# STEP 5: Lexicon sanity check
# =========================================================
POS_WORDS = {
    "جميل", "رائع", "روعه", "ممتاز", "حلو", "مميز", "مره_حلو", "يعجب", "احب", "افضل",
    "نظيف", "مرتب", "خرافي", "فخم", "ممتع", "لذيذ", "واو", "يهبل", "تحفه"
}

NEG_WORDS = {
    "سيء", "سيئ", "خايس", "زفت", "رديء", "ردي", "وصخ", "وسخ", "قذر", "غالي",
    "سيئه", "مزعج", "زحمه", "زحمة", "تعبان", "شين", "مايعجب", "سيئ جدا", "سيء جدا"
}

def lexical_sentiment_score(text: str) -> int:
    if not isinstance(text, str) or not text.strip():
        return 0
    tokens = set(text.split())
    pos_hits = sum(1 for w in POS_WORDS if w in tokens)
    neg_hits = sum(1 for w in NEG_WORDS if w in tokens)
    return pos_hits - neg_hits

def mark_contradictions(df: pd.DataFrame, label_mode: str) -> pd.DataFrame:
    df = df.copy()
    df["lex_score"] = df["text_clean"].apply(lexical_sentiment_score)
    df["is_contradictory"] = False

    if label_mode == "stars_binary":
        # y_label: negative / positive
        df.loc[(df["y_label"] == "negative") & (df["lex_score"] > 1), "is_contradictory"] = True
        df.loc[(df["y_label"] == "positive") & (df["lex_score"] < -1), "is_contradictory"] = True

    elif label_mode == "sentiment_multiclass":
        df.loc[(df["y_label"] == "negative") & (df["lex_score"] > 1), "is_contradictory"] = True
        df.loc[(df["y_label"] == "positive") & (df["lex_score"] < -1), "is_contradictory"] = True

    return df

# =========================================================
# STEP 6: Read TikTok analysis files only
# =========================================================
def find_analysis_files(root_folder: str):
    root = Path(root_folder)
    if not root.exists():
        raise FileNotFoundError(f"ROOT_FOLDER does not exist: {root_folder}")

    files = [
        p for p in root.rglob("*.xlsx")
        if p.name.endswith("_analysis.xlsx")
        and not p.name.startswith("~$")
        and "summary" not in p.name.lower()
    ]
    return sorted(files)

def read_all_analysis_files(root_folder: str) -> pd.DataFrame:
    files = find_analysis_files(root_folder)

    if not files:
        raise ValueError("No *_analysis.xlsx files were found.")

    all_dfs = []
    bad_files = []

    print(f"Found {len(files)} analysis files.")

    for fp in files:
        try:
            df = pd.read_excel(fp)

            df["Source_File"] = fp.name
            df["__path__"] = str(fp)
            df["Parent_Folder"] = fp.parent.name

            # Optional region extraction from path
            parts = fp.parts
            region_guess = None
            for part in parts:
                if "المنطقة" in part:
                    region_guess = part
                    break
            df["Region_Folder"] = region_guess

            all_dfs.append(df)

        except Exception as e:
            bad_files.append((str(fp), str(e)))

    if not all_dfs:
        raise ValueError("All files failed to load.")

    big_df = pd.concat(all_dfs, ignore_index=True)

    print(f"Loaded rows: {len(big_df):,}")
    print(f"Failed files: {len(bad_files)}")

    if bad_files:
        bad_df = pd.DataFrame(bad_files, columns=["file", "error"])
        bad_df.to_excel(os.path.join(OUTPUT_DIR, "failed_files.xlsx"), index=False)

    return big_df

# =========================================================
# STEP 7: Helper functions
# =========================================================
def pick_text_column(df: pd.DataFrame, candidates: list) -> str:
    for col in candidates:
        if col in df.columns:
            non_null_ratio = df[col].notna().mean()
            if non_null_ratio > 0:
                return col
    raise ValueError("No suitable text column found.")

def preprocess_text_column(df: pd.DataFrame, text_col: str) -> pd.DataFrame:
    df = df.copy()

    df[text_col] = df[text_col].astype(str)
    df[text_col] = df[text_col].replace(
        {"nan": "", "None": "", "null": "", "[]": "", "{}": ""}
    )

    df["TEXT_FOR_MODEL"] = df[text_col].astype(str).fillna("").str.strip()
    df = df[df["TEXT_FOR_MODEL"] != ""].copy()

    if APPLY_EXTRA_CLEANING:
        df["text_clean"] = df["TEXT_FOR_MODEL"].apply(clean_text)
    else:
        df["text_clean"] = df["TEXT_FOR_MODEL"].astype(str).str.strip()

    df["word_count"] = df["text_clean"].apply(word_count)
    df = df[df["word_count"] >= MIN_WORDS].copy()

    return df

def apply_quality_filters(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if USE_ONLY_USEFUL_COMMENTS and "is_useful" in df.columns:
        df = df[df["is_useful"] == True].copy()

    if MIN_QUALITY_SCORE is not None and "quality_score" in df.columns:
        df["quality_score"] = pd.to_numeric(df["quality_score"], errors="coerce")
        df = df[df["quality_score"] >= MIN_QUALITY_SCORE].copy()

    if MIN_COMMENT_STRENGTH is not None and "comment_strength" in df.columns:
        df["comment_strength"] = pd.to_numeric(df["comment_strength"], errors="coerce")
        df = df[df["comment_strength"] >= MIN_COMMENT_STRENGTH].copy()

    return df

def build_labels(df: pd.DataFrame, label_mode: str) -> pd.DataFrame:
    df = df.copy()

    if label_mode == "stars_binary":
        if "Stars" not in df.columns:
            raise ValueError("Column 'Stars' not found.")

        df["Stars_num"] = pd.to_numeric(df["Stars"], errors="coerce")
        df = df[df["Stars_num"].isin([1, 2, 4, 5])].copy()

        df["y_label"] = np.where(df["Stars_num"].isin([1, 2]), "negative", "positive")
        label_to_id = {"negative": 0, "positive": 1}
        df["y"] = df["y_label"].map(label_to_id)

    elif label_mode == "sentiment_multiclass":
        if "Sentiment" not in df.columns:
            raise ValueError("Column 'Sentiment' not found.")

        df["Sentiment"] = df["Sentiment"].astype(str).str.strip().str.lower()

        mapping = {
            "negative": "negative",
            "neutral": "neutral",
            "positive": "positive"
        }

        df["y_label"] = df["Sentiment"].map(mapping)
        df = df[df["y_label"].notna()].copy()

        label_to_id = {"negative": 0, "neutral": 1, "positive": 2}
        df["y"] = df["y_label"].map(label_to_id)

    else:
        raise ValueError("Invalid LABEL_MODE. Use 'stars_binary' or 'sentiment_multiclass'.")

    return df

def oversample_training_data(X_train_text, y_train):
    temp_df = pd.DataFrame({"text": X_train_text, "y": y_train})

    class_counts = temp_df["y"].value_counts()
    max_count = class_counts.max()

    balanced_parts = []
    for cls in class_counts.index:
        part = temp_df[temp_df["y"] == cls]
        part_resampled = resample(
            part,
            replace=True,
            n_samples=max_count,
            random_state=RANDOM_STATE
        )
        balanced_parts.append(part_resampled)

    balanced_df = pd.concat(balanced_parts).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    return balanced_df["text"].tolist(), balanced_df["y"].values

def compute_dynamic_max_len(texts, quantile=0.95, min_len=40, max_len=250):
    lengths = [len(str(t).split()) for t in texts if str(t).strip()]
    if not lengths:
        return 80
    q_len = int(np.quantile(lengths, quantile))
    q_len = max(min_len, q_len)
    q_len = min(max_len, q_len)
    return q_len

# =========================================================
# STEP 8: Load data
# =========================================================
df = read_all_analysis_files(ROOT_FOLDER)
print("Initial shape:", df.shape)

# =========================================================
# STEP 9: Choose text column
# =========================================================
TEXT_COL = pick_text_column(df, TEXT_COLUMN_CANDIDATES)
print("Selected text column:", TEXT_COL)

# =========================================================
# STEP 10: Apply optional quality filters
# =========================================================
df = apply_quality_filters(df)
print("After quality filters:", df.shape)

# =========================================================
# STEP 11: Text preprocessing
# =========================================================
df = preprocess_text_column(df, TEXT_COL)
print("After text preprocessing:", df.shape)

# =========================================================
# STEP 12: Build labels
# =========================================================
df = build_labels(df, LABEL_MODE)
print("After label building:", df.shape)
print("Label distribution:")
print(df["y_label"].value_counts(dropna=False))

# =========================================================
# STEP 13: Sanity check for label noise
# =========================================================
if RUN_SANITY_CHECK:
    df = mark_contradictions(df, LABEL_MODE)

    contradictions = df[df["is_contradictory"] == True].copy()
    print(f"Contradictory samples found: {len(contradictions):,}")

    if len(contradictions) > 0:
        contradictions[[
            "TEXT_FOR_MODEL", "text_clean", "y_label", "lex_score",
            "Source_File", "Parent_Folder", "Region_Folder"
        ]].to_excel(
            os.path.join(OUTPUT_DIR, f"{MODEL_NAME_PREFIX}_contradictions.xlsx"),
            index=False
        )

    if DROP_CONTRADICTORY_LABELS:
        df = df[df["is_contradictory"] == False].copy()
        print("After dropping contradictions:", df.shape)

# =========================================================
# STEP 14: Aspect extraction
# =========================================================
if RUN_ASPECT_ANALYSIS:
    df["aspects_found"] = df["text_clean"].apply(lambda x: extract_aspects(x, ASPECTS))
    df["aspects_joined"] = df["aspects_found"].apply(lambda x: ", ".join(x) if x else "")

    for aspect_name in ASPECTS.keys():
        df[f"aspect_{aspect_name}"] = df["aspects_found"].apply(lambda lst: int(aspect_name in lst))

    aspect_cols = [f"aspect_{a}" for a in ASPECTS.keys()]
    aspect_summary = pd.DataFrame({
        "aspect": list(ASPECTS.keys()),
        "frequency": [df[c].sum() for c in aspect_cols]
    }).sort_values("frequency", ascending=False)

    aspect_summary.to_excel(
        os.path.join(OUTPUT_DIR, f"{MODEL_NAME_PREFIX}_aspect_summary.xlsx"),
        index=False
    )

# =========================================================
# STEP 15: Save merged prepared dataset
# =========================================================
df.to_excel(os.path.join(OUTPUT_DIR, f"{MODEL_NAME_PREFIX}_prepared_dataset.xlsx"), index=False)

# =========================================================
# STEP 16: Train / Val / Test split
# =========================================================
X = df["text_clean"].astype(str).tolist()
y = df["y"].values

X_train_full, X_test, y_train_full, y_test, idx_train_full, idx_test = train_test_split(
    X, y, df.index.values,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

val_ratio_adjusted = VAL_SIZE / (1 - TEST_SIZE)

X_train, X_val, y_train, y_val, idx_train, idx_val = train_test_split(
    X_train_full, y_train_full, idx_train_full,
    test_size=val_ratio_adjusted,
    random_state=RANDOM_STATE,
    stratify=y_train_full
)

print("Train size:", len(X_train))
print("Val size:", len(X_val))
print("Test size:", len(X_test))

# =========================================================
# STEP 17: Optional balancing
# =========================================================
class_weight_dict = None

if USE_OVERSAMPLING:
    X_train, y_train = oversample_training_data(X_train, y_train)
    print("After oversampling train distribution:", Counter(y_train))

elif USE_CLASS_WEIGHTS:
    classes = np.unique(y_train)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
    class_weight_dict = {int(c): float(w) for c, w in zip(classes, weights)}
    print("Class weights:", class_weight_dict)

# =========================================================
# STEP 18: Tokenization and padding
# =========================================================
MAX_LEN = compute_dynamic_max_len(
    texts=X_train,
    quantile=MAX_LEN_QUANTILE,
    min_len=MIN_MAX_LEN,
    max_len=MAX_MAX_LEN
)

print("Dynamic MAX_LEN:", MAX_LEN)

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq   = tokenizer.texts_to_sequences(X_val)
X_test_seq  = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_val_pad   = pad_sequences(X_val_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_test_pad  = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding="post", truncating="post")

# =========================================================
# STEP 19: Build model
# =========================================================
num_classes = len(np.unique(y))

model = Sequential()
model.add(Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM, input_length=MAX_LEN))
model.add(SpatialDropout1D(0.2))
model.add(Bidirectional(LSTM(LSTM_UNITS, dropout=0.2, recurrent_dropout=0.2)))
model.add(Dense(DENSE_UNITS, activation="relu"))
model.add(Dropout(DROPOUT_RATE))

if LABEL_MODE == "stars_binary":
    model.add(Dense(1, activation="sigmoid"))
    model.compile(
        loss="binary_crossentropy",
        optimizer="adam",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
    )
else:
    model.add(Dense(num_classes, activation="softmax"))
    model.compile(
        loss="sparse_categorical_crossentropy",
        optimizer="adam",
        metrics=["accuracy"]
    )

model.summary()

# =========================================================
# STEP 20: Callbacks
# =========================================================
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=os.path.join(OUTPUT_DIR, f"{MODEL_NAME_PREFIX}_best.keras"),
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
]

# =========================================================
# STEP 21: Train
# =========================================================
history = model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_val_pad, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1
)

# =========================================================
# STEP 22: Evaluate
# =========================================================
if LABEL_MODE == "stars_binary":
    y_prob = model.predict(X_test_pad, batch_size=BATCH_SIZE, verbose=1).ravel()
    y_pred = (y_prob >= 0.5).astype(int)

    label_names = ["negative", "positive"]

    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average="macro")
    f1_weighted = f1_score(y_test, y_pred, average="weighted")

    report_dict = classification_report(
        y_test, y_pred,
        target_names=label_names,
        output_dict=True
    )
    cm = confusion_matrix(y_test, y_pred)

    metrics_summary = {
        "accuracy": float(acc),
        "f1_macro": float(f1_macro),
        "f1_weighted": float(f1_weighted)
    }

else:
    y_prob = model.predict(X_test_pad, batch_size=BATCH_SIZE, verbose=1)
    y_pred = np.argmax(y_prob, axis=1)

    label_names = ["negative", "neutral", "positive"]

    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average="macro")
    f1_weighted = f1_score(y_test, y_pred, average="weighted")

    report_dict = classification_report(
        y_test, y_pred,
        target_names=label_names,
        output_dict=True
    )
    cm = confusion_matrix(y_test, y_pred)

    metrics_summary = {
        "accuracy": float(acc),
        "f1_macro": float(f1_macro),
        "f1_weighted": float(f1_weighted)
    }

print("\nAccuracy:", acc)
print("F1 Macro:", f1_macro)
print("F1 Weighted:", f1_weighted)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=label_names))

print("\nConfusion Matrix:")
print(cm)

# =========================================================
# STEP 23: Save metrics and reports
# =========================================================
with open(os.path.join(OUTPUT_DIR, f"{MODEL_NAME_PREFIX}_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(metrics_summary, f, ensure_ascii=False, indent=2)

report_df = pd.DataFrame(report_dict).transpose()
report_df.to_excel(os.path.join(OUTPUT_DIR, f"{MODEL_NAME_PREFIX}_classification_report.xlsx"))

cm_df = pd.DataFrame(cm, index=label_names, columns=label_names)
cm_df.to_excel(os.path.join(OUTPUT_DIR, f"{MODEL_NAME_PREFIX}_confusion_matrix.xlsx"))

history_df = pd.DataFrame(history.history)
history_df.to_excel(os.path.join(OUTPUT_DIR, f"{MODEL_NAME_PREFIX}_history.xlsx"), index=False)

# =========================================================
# STEP 24: Error analysis
# =========================================================
test_meta = df.loc[idx_test].copy().reset_index(drop=True)
test_meta["y_true"] = y_test
test_meta["y_pred"] = y_pred
test_meta["true_label"] = test_meta["y_true"].map({i: n for i, n in enumerate(label_names)})
test_meta["pred_label"] = test_meta["y_pred"].map({i: n for i, n in enumerate(label_names)})

if LABEL_MODE == "stars_binary":
    test_meta["pred_confidence"] = y_prob
else:
    test_meta["pred_confidence"] = y_prob.max(axis=1)

errors_df = test_meta[test_meta["y_true"] != test_meta["y_pred"]].copy()
errors_df = errors_df.sort_values("pred_confidence", ascending=False)

cols_to_export = [
    "TEXT_FOR_MODEL", "text_clean", "true_label", "pred_label", "pred_confidence",
    "Source_File", "Parent_Folder", "Region_Folder"
]

extra_cols = [
    "Sentiment", "Stars", "quality_score", "comment_strength",
    "is_useful", "diggCount", "replyCommentTotal", "videoWebUrl", "places", "aspects_joined"
]

for c in extra_cols:
    if c in errors_df.columns and c not in cols_to_export:
        cols_to_export.append(c)

errors_df[cols_to_export].to_excel(
    os.path.join(OUTPUT_DIR, f"{MODEL_NAME_PREFIX}_error_analysis.xlsx"),
    index=False
)

# =========================================================
# STEP 25: Save predictions on test set
# =========================================================
test_export = test_meta.copy()

if LABEL_MODE == "stars_binary":
    test_export["prob_positive"] = y_prob
else:
    for i, label in enumerate(label_names):
        test_export[f"prob_{label}"] = y_prob[:, i]

test_export.to_excel(
    os.path.join(OUTPUT_DIR, f"{MODEL_NAME_PREFIX}_test_predictions.xlsx"),
    index=False
)

# =========================================================
# STEP 26: Save model + tokenizer + config
# =========================================================
model.save(os.path.join(OUTPUT_DIR, f"{MODEL_NAME_PREFIX}_final.keras"))

with open(os.path.join(OUTPUT_DIR, f"{MODEL_NAME_PREFIX}_tokenizer.pkl"), "wb") as f:
    pickle.dump(tokenizer, f)

config = {
    "ROOT_FOLDER": ROOT_FOLDER,
    "OUTPUT_DIR": OUTPUT_DIR,
    "LABEL_MODE": LABEL_MODE,
    "TEXT_COL": TEXT_COL,
    "TEXT_COLUMN_CANDIDATES": TEXT_COLUMN_CANDIDATES,
    "USE_ONLY_USEFUL_COMMENTS": USE_ONLY_USEFUL_COMMENTS,
    "MIN_QUALITY_SCORE": MIN_QUALITY_SCORE,
    "MIN_COMMENT_STRENGTH": MIN_COMMENT_STRENGTH,
    "APPLY_EXTRA_CLEANING": APPLY_EXTRA_CLEANING,
    "MIN_WORDS": MIN_WORDS,
    "TEST_SIZE": TEST_SIZE,
    "VAL_SIZE": VAL_SIZE,
    "VOCAB_SIZE": VOCAB_SIZE,
    "EMBEDDING_DIM": EMBEDDING_DIM,
    "LSTM_UNITS": LSTM_UNITS,
    "DENSE_UNITS": DENSE_UNITS,
    "DROPOUT_RATE": DROPOUT_RATE,
    "BATCH_SIZE": BATCH_SIZE,
    "EPOCHS": EPOCHS,
    "RANDOM_STATE": RANDOM_STATE,
    "MAX_LEN": MAX_LEN,
    "MAX_LEN_QUANTILE": MAX_LEN_QUANTILE,
    "MIN_MAX_LEN": MIN_MAX_LEN,
    "MAX_MAX_LEN": MAX_MAX_LEN,
    "USE_OVERSAMPLING": USE_OVERSAMPLING,
    "USE_CLASS_WEIGHTS": USE_CLASS_WEIGHTS,
    "RUN_SANITY_CHECK": RUN_SANITY_CHECK,
    "DROP_CONTRADICTORY_LABELS": DROP_CONTRADICTORY_LABELS,
    "RUN_ASPECT_ANALYSIS": RUN_ASPECT_ANALYSIS,
    "LABEL_NAMES": label_names
}

with open(os.path.join(OUTPUT_DIR, f"{MODEL_NAME_PREFIX}_config.json"), "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

# =========================================================
# STEP 27: Final summary
# =========================================================
summary = {
    "total_rows_after_preparation": int(len(df)),
    "train_size": int(len(X_train)),
    "val_size": int(len(X_val)),
    "test_size": int(len(X_test)),
    "text_column_used": TEXT_COL,
    "label_mode": LABEL_MODE,
    "label_distribution": {str(k): int(v) for k, v in df["y_label"].value_counts().to_dict().items()},
    "max_len": int(MAX_LEN),
    "vocab_size": VOCAB_SIZE,
    "accuracy": float(acc),
    "f1_macro": float(f1_macro),
    "f1_weighted": float(f1_weighted)
}

with open(os.path.join(OUTPUT_DIR, f"{MODEL_NAME_PREFIX}_summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("\nDone successfully.")
print(json.dumps(summary, ensure_ascii=False, indent=2))

2026-04-14 07:44:32.330916: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776152672.531290      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776152672.589656      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776152673.030767      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776152673.030805      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776152673.030807      55 computation_placer.cc:177] computation placer alr

Found 296 analysis files.
Loaded rows: 7,280
Failed files: 0
Initial shape: (7280, 80)
Selected text column: Text_TR
After quality filters: (7280, 80)
After text preprocessing: (7280, 83)
After label building: (7280, 85)
Label distribution:
y_label
positive    3871
negative    2159
neutral     1250
Name: count, dtype: int64
Contradictory samples found: 4
Train size: 5824
Val size: 728
Test size: 728
Class weights: {0: 1.1241073151901178, 1: 1.9413333333333334, 2: 0.6268431815735658}
Dynamic MAX_LEN: 40


I0000 00:00:1776152717.499340      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1776152717.505627      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 163ms/step - accuracy: 0.3692 - loss: 1.0965
Epoch 1: val_loss improved from inf to 1.03035, saving model to /kaggle/working/tiktok_lstm_output/tiktok_lstm_sentiment_multiclass_best.keras
23/23 ━━━━━━━━━━━━━━━━━━━━ 15s 221ms/step - accuracy: 0.3707 - loss: 1.0961 - val_accuracy: 0.5275 - val_loss: 1.0304 - learning_rate: 0.0010
Epoch 2/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 167ms/step - accuracy: 0.5495 - loss: 0.9742
Epoch 2: val_loss improved from 1.03035 to 0.77364, saving model to /kaggle/working/tiktok_lstm_output/tiktok_lstm_sentiment_multiclass_best.keras
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 192ms/step - accuracy: 0.5520 - loss: 0.9709 - val_accuracy: 0.6703 - val_loss: 0.7736 - learning_rate: 0.0010
Epoch 3/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 163ms/step - accuracy: 0.8044 - loss: 0.5238
Epoch 3: val_loss improved from 0.77364 to 0.71066, saving model to /kaggle/working/tiktok_lstm_output/tiktok_lstm_sentiment_multiclass_best.keras
23/23 ━━━━━━━━━━━━━━━━

In [4]:
# =========================================================
# TikTok LSTM Sentiment Model - Binary Only
# Uses Stars only:
# 1,2 = negative
# 4,5 = positive
# 3   = removed
# Reads only *_analysis.xlsx files
# Saves everything to /kaggle/working/
# =========================================================

# =========================================================
# STEP 0: Imports
# =========================================================
import os
import re
import json
import pickle
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    roc_auc_score
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.utils import resample

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SpatialDropout1D, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

warnings.filterwarnings("ignore")

# =========================================================
# STEP 1: Settings
# =========================================================
ROOT_FOLDER = r"/kaggle/input/datasets/ziyadaltalhi/tik-tok-data/Tik Tok Datasets - After Processing"
OUTPUT_DIR = r"/kaggle/working/tiktok_lstm_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

LABEL_MODE = "stars_binary"

TEXT_COLUMN_CANDIDATES = [
    "Text_TR",
    "Text_ML",
    "Text_Cleaned",
    "Text_Normalized",
    "Text",
    "Text_Orig",
    "text"
]

# Optional filters
USE_ONLY_USEFUL_COMMENTS = False
MIN_QUALITY_SCORE = None        # Example: 6
MIN_COMMENT_STRENGTH = None     # Example: 5

# Text processing
APPLY_EXTRA_CLEANING = True
MIN_WORDS = 2

# Training settings
TEST_SIZE = 0.10
VAL_SIZE = 0.10
VOCAB_SIZE = 50000
EMBEDDING_DIM = 128
LSTM_UNITS = 128
DENSE_UNITS = 64
DROPOUT_RATE = 0.30
BATCH_SIZE = 256
EPOCHS = 15
RANDOM_STATE = 42

# Sequence length
MAX_LEN_QUANTILE = 0.95
MIN_MAX_LEN = 40
MAX_MAX_LEN = 250

# Balancing
USE_OVERSAMPLING = False
USE_CLASS_WEIGHTS = True

# Label noise check
RUN_SANITY_CHECK = True
DROP_CONTRADICTORY_LABELS = False

# Aspect extraction
RUN_ASPECT_ANALYSIS = True

MODEL_NAME_PREFIX = "tiktok_lstm_stars_binary"

# =========================================================
# STEP 2: Reproducibility
# =========================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(RANDOM_STATE)

# =========================================================
# STEP 3: Text cleaning
# =========================================================
AR_DIACRITICS = re.compile(r'[\u0617-\u061A\u064B-\u0652]')
URL_RE = re.compile(r'https?://\S+|www\.\S+')
EMAIL_RE = re.compile(r'\S+@\S+')
MENTION_RE = re.compile(r'@\w+')
HASHTAG_RE = re.compile(r'#(\w+)')
MULTISPACE_RE = re.compile(r'\s+')
REPEAT_CHARS_RE = re.compile(r'(.)\1{2,}')

def normalize_arabic(text: str) -> str:
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"ئ", "ي", text)
    text = re.sub(r"ة", "ه", text)
    text = re.sub(r"ـ", "", text)
    text = AR_DIACRITICS.sub("", text)
    return text

def clean_text(text: str) -> str:
    if pd.isna(text):
        return ""

    text = str(text).strip().lower()

    if text in {"", "nan", "none", "null", "[]", "{}"}:
        return ""

    text = URL_RE.sub(" ", text)
    text = EMAIL_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    text = HASHTAG_RE.sub(r" \1 ", text)

    text = normalize_arabic(text)

    # Keep Arabic, English, digits, spaces
    text = re.sub(r"[^0-9a-zA-Z\u0600-\u06FF\s]", " ", text)

    # Reduce repeated letters
    text = REPEAT_CHARS_RE.sub(r"\1\1", text)

    text = MULTISPACE_RE.sub(" ", text).strip()
    return text

def word_count(text: str) -> int:
    if not isinstance(text, str) or not text.strip():
        return 0
    return len(text.split())

# =========================================================
# STEP 4: Aspect keywords
# =========================================================
ASPECTS = {
    "المنظر_والطبيعه": [
        "منظر", "اطلاله", "اطلالة", "طبيعه", "طبيعة", "جميل", "روعه", "روعة"
    ],
    "الاسعار": [
        "سعر", "اسعار", "غالي", "رخيص", "مبالغ", "مبالغه", "مبالغة"
    ],
    "النظافه": [
        "نظيف", "نظافه", "نظافة", "وسخ", "قذر", "متسخ"
    ],
    "الخدمه": [
        "خدمه", "خدمة", "تعامل", "موظف", "موظفين", "استقبال", "خدمات"
    ],
    "المواقف": [
        "موقف", "مواقف", "سيارات", "باركنج", "parking"
    ],
    "الازدحام": [
        "زحمه", "زحمة", "ازدحام", "مزدحم", "هدوء", "زحام"
    ],
    "الاجواء": [
        "اجواء", "أجواء", "جو", "الجو", "بارد", "حار", "حر", "معتدل"
    ],
    "الموقع": [
        "موقع", "الموقع", "مكان", "بعيد", "قريب", "سهل", "صعب"
    ]
}

def extract_aspects(text: str, aspects_dict: dict) -> list:
    found = []
    if not isinstance(text, str) or not text.strip():
        return found

    for aspect_name, keywords in aspects_dict.items():
        for kw in keywords:
            if re.search(rf"\b{re.escape(kw)}\b", text):
                found.append(aspect_name)
                break
    return found

# =========================================================
# STEP 5: Lexicon sanity check
# =========================================================
POS_WORDS = {
    "جميل", "رائع", "روعه", "ممتاز", "حلو", "مميز", "يعجب", "احب", "افضل",
    "نظيف", "مرتب", "خرافي", "فخم", "ممتع", "لذيذ", "واو", "يهبل", "تحفه"
}

NEG_WORDS = {
    "سيء", "سيئ", "خايس", "زفت", "رديء", "ردي", "وصخ", "وسخ", "قذر", "غالي",
    "سيئه", "مزعج", "زحمه", "زحمة", "تعبان", "شين", "مايعجب", "سيئ جدا", "سيء جدا"
}

def lexical_sentiment_score(text: str) -> int:
    if not isinstance(text, str) or not text.strip():
        return 0
    tokens = set(text.split())
    pos_hits = sum(1 for w in POS_WORDS if w in tokens)
    neg_hits = sum(1 for w in NEG_WORDS if w in tokens)
    return pos_hits - neg_hits

def mark_contradictions(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["lex_score"] = df["text_clean"].apply(lexical_sentiment_score)
    df["is_contradictory"] = False

    df.loc[(df["y_label"] == "negative") & (df["lex_score"] > 1), "is_contradictory"] = True
    df.loc[(df["y_label"] == "positive") & (df["lex_score"] < -1), "is_contradictory"] = True

    return df

# =========================================================
# STEP 6: Read TikTok analysis files only
# =========================================================
def find_analysis_files(root_folder: str):
    root = Path(root_folder)
    if not root.exists():
        raise FileNotFoundError(f"ROOT_FOLDER does not exist: {root_folder}")

    files = [
        p for p in root.rglob("*.xlsx")
        if p.name.endswith("_analysis.xlsx")
        and not p.name.startswith("~$")
        and "summary" not in p.name.lower()
    ]
    return sorted(files)

def read_all_analysis_files(root_folder: str) -> pd.DataFrame:
    files = find_analysis_files(root_folder)

    if not files:
        raise ValueError("No *_analysis.xlsx files were found.")

    all_dfs = []
    bad_files = []

    print(f"Found {len(files)} analysis files.")

    for fp in files:
        try:
            df = pd.read_excel(fp)

            df["Source_File"] = fp.name
            df["__path__"] = str(fp)
            df["Parent_Folder"] = fp.parent.name

            region_guess = None
            for part in fp.parts:
                if "المنطقة" in part:
                    region_guess = part
                    break
            df["Region_Folder"] = region_guess

            all_dfs.append(df)

        except Exception as e:
            bad_files.append((str(fp), str(e)))

    if not all_dfs:
        raise ValueError("All files failed to load.")

    big_df = pd.concat(all_dfs, ignore_index=True)

    print(f"Loaded rows: {len(big_df):,}")
    print(f"Failed files: {len(bad_files)}")

    if bad_files:
        bad_df = pd.DataFrame(bad_files, columns=["file", "error"])
        bad_df.to_excel(os.path.join(OUTPUT_DIR, "failed_files.xlsx"), index=False)

    return big_df

# =========================================================
# STEP 7: Helper functions
# =========================================================
def pick_text_column(df: pd.DataFrame, candidates: list) -> str:
    best_col = None
    best_non_empty = -1

    for col in candidates:
        if col in df.columns:
            non_empty = df[col].astype(str).str.strip().replace("nan", "").ne("").sum()
            if non_empty > best_non_empty:
                best_non_empty = non_empty
                best_col = col

    if best_col is None:
        raise ValueError("No suitable text column found.")

    return best_col

def apply_quality_filters(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if USE_ONLY_USEFUL_COMMENTS and "is_useful" in df.columns:
        useful_mask = df["is_useful"].astype(str).str.lower().isin(["true", "1", "yes"])
        df = df[useful_mask].copy()

    if MIN_QUALITY_SCORE is not None and "quality_score" in df.columns:
        df["quality_score"] = pd.to_numeric(df["quality_score"], errors="coerce")
        df = df[df["quality_score"] >= MIN_QUALITY_SCORE].copy()

    if MIN_COMMENT_STRENGTH is not None and "comment_strength" in df.columns:
        df["comment_strength"] = pd.to_numeric(df["comment_strength"], errors="coerce")
        df = df[df["comment_strength"] >= MIN_COMMENT_STRENGTH].copy()

    return df

def preprocess_text_column(df: pd.DataFrame, text_col: str) -> pd.DataFrame:
    df = df.copy()

    df[text_col] = df[text_col].fillna("").astype(str).replace(
        {"nan": "", "None": "", "null": "", "[]": "", "{}": ""}
    )

    df["TEXT_FOR_MODEL"] = df[text_col].astype(str).str.strip()
    df = df[df["TEXT_FOR_MODEL"] != ""].copy()

    if APPLY_EXTRA_CLEANING:
        df["text_clean"] = df["TEXT_FOR_MODEL"].apply(clean_text)
    else:
        df["text_clean"] = df["TEXT_FOR_MODEL"].astype(str).str.strip()

    df["word_count"] = df["text_clean"].apply(word_count)
    df = df[df["word_count"] >= MIN_WORDS].copy()

    return df

def build_binary_labels_from_stars(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "Stars" not in df.columns:
        raise ValueError("Column 'Stars' not found in the analysis files.")

    df["Stars_num"] = pd.to_numeric(df["Stars"], errors="coerce")
    df = df[df["Stars_num"].isin([1, 2, 4, 5])].copy()

    df["y_label"] = np.where(df["Stars_num"].isin([1, 2]), "negative", "positive")
    label_to_id = {"negative": 0, "positive": 1}
    df["y"] = df["y_label"].map(label_to_id)

    return df

def oversample_training_data(X_train_text, y_train):
    temp_df = pd.DataFrame({"text": X_train_text, "y": y_train})

    class_counts = temp_df["y"].value_counts()
    max_count = class_counts.max()

    balanced_parts = []
    for cls in class_counts.index:
        part = temp_df[temp_df["y"] == cls]
        part_resampled = resample(
            part,
            replace=True,
            n_samples=max_count,
            random_state=RANDOM_STATE
        )
        balanced_parts.append(part_resampled)

    balanced_df = pd.concat(balanced_parts).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    return balanced_df["text"].tolist(), balanced_df["y"].values

def compute_dynamic_max_len(texts, quantile=0.95, min_len=40, max_len=250):
    lengths = [len(str(t).split()) for t in texts if str(t).strip()]
    if not lengths:
        return 80
    q_len = int(np.quantile(lengths, quantile))
    q_len = max(min_len, q_len)
    q_len = min(max_len, q_len)
    return q_len

# =========================================================
# STEP 8: Load data
# =========================================================
df = read_all_analysis_files(ROOT_FOLDER)
print("Initial shape:", df.shape)

# =========================================================
# STEP 9: Choose text column
# =========================================================
TEXT_COL = pick_text_column(df, TEXT_COLUMN_CANDIDATES)
print("Selected text column:", TEXT_COL)

# =========================================================
# STEP 10: Apply quality filters
# =========================================================
df = apply_quality_filters(df)
print("After quality filters:", df.shape)

# =========================================================
# STEP 11: Text preprocessing
# =========================================================
df = preprocess_text_column(df, TEXT_COL)
print("After text preprocessing:", df.shape)

# =========================================================
# STEP 12: Build binary labels from Stars
# =========================================================
df = build_binary_labels_from_stars(df)
print("After label building:", df.shape)
print("Label distribution:")
print(df["y_label"].value_counts(dropna=False))

# =========================================================
# STEP 13: Sanity check
# =========================================================
if RUN_SANITY_CHECK:
    df = mark_contradictions(df)

    contradictions = df[df["is_contradictory"] == True].copy()
    print(f"Contradictory samples found: {len(contradictions):,}")

    if len(contradictions) > 0:
        contradictions_cols = [
            "TEXT_FOR_MODEL", "text_clean", "y_label", "lex_score",
            "Source_File", "Parent_Folder", "Region_Folder"
        ]
        contradictions[contradictions_cols].to_excel(
            os.path.join(OUTPUT_DIR, f"{MODEL_NAME_PREFIX}_contradictions.xlsx"),
            index=False
        )

    if DROP_CONTRADICTORY_LABELS:
        df = df[df["is_contradictory"] == False].copy()
        print("After dropping contradictions:", df.shape)

# =========================================================
# STEP 14: Aspect extraction
# =========================================================
if RUN_ASPECT_ANALYSIS:
    df["aspects_found"] = df["text_clean"].apply(lambda x: extract_aspects(x, ASPECTS))
    df["aspects_joined"] = df["aspects_found"].apply(lambda x: ", ".join(x) if x else "")

    for aspect_name in ASPECTS.keys():
        df[f"aspect_{aspect_name}"] = df["aspects_found"].apply(lambda lst: int(aspect_name in lst))

    aspect_cols = [f"aspect_{a}" for a in ASPECTS.keys()]
    aspect_summary = pd.DataFrame({
        "aspect": list(ASPECTS.keys()),
        "frequency": [int(df[c].sum()) for c in aspect_cols]
    }).sort_values("frequency", ascending=False)

    aspect_summary.to_excel(
        os.path.join(OUTPUT_DIR, f"{MODEL_NAME_PREFIX}_aspect_summary.xlsx"),
        index=False
    )

# =========================================================
# STEP 15: Save prepared dataset
# =========================================================
df.to_excel(
    os.path.join(OUTPUT_DIR, f"{MODEL_NAME_PREFIX}_prepared_dataset.xlsx"),
    index=False
)

# =========================================================
# STEP 16: Train / Val / Test split
# =========================================================
X = df["text_clean"].astype(str).tolist()
y = df["y"].values

X_train_full, X_test, y_train_full, y_test, idx_train_full, idx_test = train_test_split(
    X, y, df.index.values,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

val_ratio_adjusted = VAL_SIZE / (1 - TEST_SIZE)

X_train, X_val, y_train, y_val, idx_train, idx_val = train_test_split(
    X_train_full, y_train_full, idx_train_full,
    test_size=val_ratio_adjusted,
    random_state=RANDOM_STATE,
    stratify=y_train_full
)

print("Train size:", len(X_train))
print("Val size:", len(X_val))
print("Test size:", len(X_test))

# =========================================================
# STEP 17: Balancing
# =========================================================
class_weight_dict = None

if USE_OVERSAMPLING:
    X_train, y_train = oversample_training_data(X_train, y_train)
    print("After oversampling train distribution:", Counter(y_train))

elif USE_CLASS_WEIGHTS:
    classes = np.unique(y_train)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
    class_weight_dict = {int(c): float(w) for c, w in zip(classes, weights)}
    print("Class weights:", class_weight_dict)

# =========================================================
# STEP 18: Tokenization and padding
# =========================================================
MAX_LEN = compute_dynamic_max_len(
    texts=X_train,
    quantile=MAX_LEN_QUANTILE,
    min_len=MIN_MAX_LEN,
    max_len=MAX_MAX_LEN
)

print("Dynamic MAX_LEN:", MAX_LEN)

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq   = tokenizer.texts_to_sequences(X_val)
X_test_seq  = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_val_pad   = pad_sequences(X_val_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_test_pad  = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding="post", truncating="post")

# =========================================================
# STEP 19: Build binary model
# =========================================================
model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM),
    SpatialDropout1D(0.2),
    Bidirectional(LSTM(LSTM_UNITS, dropout=0.2, recurrent_dropout=0.2)),
    Dense(DENSE_UNITS, activation="relu"),
    Dropout(DROPOUT_RATE),
    Dense(1, activation="sigmoid")
])

model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)

model.build(input_shape=(None, MAX_LEN))
model.summary()

# =========================================================
# STEP 20: Callbacks
# =========================================================
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=os.path.join(OUTPUT_DIR, f"{MODEL_NAME_PREFIX}_best.keras"),
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
]

# =========================================================
# STEP 21: Train
# =========================================================
history = model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_val_pad, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1
)

# =========================================================
# STEP 22: Evaluate
# =========================================================
y_prob = model.predict(X_test_pad, batch_size=BATCH_SIZE, verbose=1).ravel()
y_pred = (y_prob >= 0.5).astype(int)

label_names = ["negative", "positive"]

acc = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")
f1_weighted = f1_score(y_test, y_pred, average="weighted")

try:
    roc_auc = roc_auc_score(y_test, y_prob)
except:
    roc_auc = None

report_dict = classification_report(
    y_test, y_pred,
    target_names=label_names,
    output_dict=True
)
cm = confusion_matrix(y_test, y_pred)

print("\nAccuracy:", acc)
print("F1 Macro:", f1_macro)
print("F1 Weighted:", f1_weighted)
print("ROC AUC:", roc_auc)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=label_names))

print("\nConfusion Matrix:")
print(cm)

# =========================================================
# STEP 23: Save metrics and reports
# =========================================================
metrics_summary = {
    "accuracy": float(acc),
    "f1_macro": float(f1_macro),
    "f1_weighted": float(f1_weighted),
    "roc_auc": None if roc_auc is None else float(roc_auc)
}

with open(os.path.join(OUTPUT_DIR, f"{MODEL_NAME_PREFIX}_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(metrics_summary, f, ensure_ascii=False, indent=2)

report_df = pd.DataFrame(report_dict).transpose()
report_df.to_excel(os.path.join(OUTPUT_DIR, f"{MODEL_NAME_PREFIX}_classification_report.xlsx"))

cm_df = pd.DataFrame(cm, index=label_names, columns=label_names)
cm_df.to_excel(os.path.join(OUTPUT_DIR, f"{MODEL_NAME_PREFIX}_confusion_matrix.xlsx"))

history_df = pd.DataFrame(history.history)
history_df.to_excel(os.path.join(OUTPUT_DIR, f"{MODEL_NAME_PREFIX}_history.xlsx"), index=False)

# =========================================================
# STEP 24: Error analysis
# =========================================================
test_meta = df.loc[idx_test].copy().reset_index(drop=True)
test_meta["y_true"] = y_test
test_meta["y_pred"] = y_pred
test_meta["true_label"] = test_meta["y_true"].map({0: "negative", 1: "positive"})
test_meta["pred_label"] = test_meta["y_pred"].map({0: "negative", 1: "positive"})
test_meta["prob_positive"] = y_prob
test_meta["pred_confidence"] = np.where(y_pred == 1, y_prob, 1 - y_prob)

errors_df = test_meta[test_meta["y_true"] != test_meta["y_pred"]].copy()
errors_df = errors_df.sort_values("pred_confidence", ascending=False)

cols_to_export = [
    "TEXT_FOR_MODEL", "text_clean", "true_label", "pred_label",
    "prob_positive", "pred_confidence",
    "Source_File", "Parent_Folder", "Region_Folder"
]

extra_cols = [
    "Sentiment", "Stars", "quality_score", "comment_strength",
    "is_useful", "diggCount", "replyCommentTotal", "videoWebUrl", "places", "aspects_joined"
]

for c in extra_cols:
    if c in errors_df.columns and c not in cols_to_export:
        cols_to_export.append(c)

errors_df[cols_to_export].to_excel(
    os.path.join(OUTPUT_DIR, f"{MODEL_NAME_PREFIX}_error_analysis.xlsx"),
    index=False
)

# =========================================================
# STEP 25: Save test predictions
# =========================================================
test_export = test_meta.copy()
test_export.to_excel(
    os.path.join(OUTPUT_DIR, f"{MODEL_NAME_PREFIX}_test_predictions.xlsx"),
    index=False
)

# =========================================================
# STEP 26: Save model + tokenizer + config
# =========================================================
model.save(os.path.join(OUTPUT_DIR, f"{MODEL_NAME_PREFIX}_final.keras"))

with open(os.path.join(OUTPUT_DIR, f"{MODEL_NAME_PREFIX}_tokenizer.pkl"), "wb") as f:
    pickle.dump(tokenizer, f)

config = {
    "ROOT_FOLDER": ROOT_FOLDER,
    "OUTPUT_DIR": OUTPUT_DIR,
    "LABEL_MODE": LABEL_MODE,
    "TEXT_COL": TEXT_COL,
    "TEXT_COLUMN_CANDIDATES": TEXT_COLUMN_CANDIDATES,
    "USE_ONLY_USEFUL_COMMENTS": USE_ONLY_USEFUL_COMMENTS,
    "MIN_QUALITY_SCORE": MIN_QUALITY_SCORE,
    "MIN_COMMENT_STRENGTH": MIN_COMMENT_STRENGTH,
    "APPLY_EXTRA_CLEANING": APPLY_EXTRA_CLEANING,
    "MIN_WORDS": MIN_WORDS,
    "TEST_SIZE": TEST_SIZE,
    "VAL_SIZE": VAL_SIZE,
    "VOCAB_SIZE": VOCAB_SIZE,
    "EMBEDDING_DIM": EMBEDDING_DIM,
    "LSTM_UNITS": LSTM_UNITS,
    "DENSE_UNITS": DENSE_UNITS,
    "DROPOUT_RATE": DROPOUT_RATE,
    "BATCH_SIZE": BATCH_SIZE,
    "EPOCHS": EPOCHS,
    "RANDOM_STATE": RANDOM_STATE,
    "MAX_LEN": int(MAX_LEN),
    "MAX_LEN_QUANTILE": MAX_LEN_QUANTILE,
    "MIN_MAX_LEN": MIN_MAX_LEN,
    "MAX_MAX_LEN": MAX_MAX_LEN,
    "USE_OVERSAMPLING": USE_OVERSAMPLING,
    "USE_CLASS_WEIGHTS": USE_CLASS_WEIGHTS,
    "RUN_SANITY_CHECK": RUN_SANITY_CHECK,
    "DROP_CONTRADICTORY_LABELS": DROP_CONTRADICTORY_LABELS,
    "RUN_ASPECT_ANALYSIS": RUN_ASPECT_ANALYSIS,
    "LABEL_NAMES": label_names
}

with open(os.path.join(OUTPUT_DIR, f"{MODEL_NAME_PREFIX}_config.json"), "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

# =========================================================
# STEP 27: Final summary
# =========================================================
summary = {
    "total_rows_after_preparation": int(len(df)),
    "train_size": int(len(X_train)),
    "val_size": int(len(X_val)),
    "test_size": int(len(X_test)),
    "text_column_used": TEXT_COL,
    "label_mode": LABEL_MODE,
    "label_distribution": {str(k): int(v) for k, v in df["y_label"].value_counts().to_dict().items()},
    "max_len": int(MAX_LEN),
    "vocab_size": VOCAB_SIZE,
    "accuracy": float(acc),
    "f1_macro": float(f1_macro),
    "f1_weighted": float(f1_weighted),
    "roc_auc": None if roc_auc is None else float(roc_auc)
}

with open(os.path.join(OUTPUT_DIR, f"{MODEL_NAME_PREFIX}_summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("\nDone successfully.")
print(json.dumps(summary, ensure_ascii=False, indent=2))

Found 296 analysis files.
Loaded rows: 7,280
Failed files: 0
Initial shape: (7280, 80)
Selected text column: Text_TR
After quality filters: (7280, 80)
After text preprocessing: (7280, 83)
After label building: (6030, 86)
Label distribution:
y_label
positive    3871
negative    2159
Name: count, dtype: int64
Contradictory samples found: 4
Train size: 4824
Val size: 603
Test size: 603
Class weights: {0: 1.396641574985524, 1: 0.778818211172102}
Dynamic MAX_LEN: 40


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 40, 128)        │     6,400,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_1             │ (None, 40, 128)        │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 256)            │       263,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,679,681 (25.48 MB)

 Trainable params: 6,679,681 (25.48 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 164ms/step - accuracy: 0.6347 - auc: 0.5527 - loss: 0.6865
Epoch 1: val_loss improved from inf to 0.66244, saving model to /kaggle/working/tiktok_lstm_output/tiktok_lstm_stars_binary_best.keras
19/19 ━━━━━━━━━━━━━━━━━━━━ 12s 241ms/step - accuracy: 0.6351 - auc: 0.5556 - loss: 0.6863 - val_accuracy: 0.6468 - val_auc: 0.7506 - val_loss: 0.6624 - learning_rate: 0.0010
Epoch 2/15
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 165ms/step - accuracy: 0.7552 - auc: 0.8337 - loss: 0.5638
Epoch 2: val_loss improved from 0.66244 to 0.51749, saving model to /kaggle/working/tiktok_lstm_output/tiktok_lstm_stars_binary_best.keras
19/19 ━━━━━━━━━━━━━━━━━━━━ 4s 194ms/step - accuracy: 0.7569 - auc: 0.8356 - loss: 0.5602 - val_accuracy: 0.7794 - val_auc: 0.8646 - val_loss: 0.5175 - learning_rate: 0.0010
Epoch 3/15
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step - accuracy: 0.8784 - auc: 0.9469 - loss: 0.2960
Epoch 3: val_loss improved from 0.51749 to 0.46762, saving model to /kaggle/wor

In [6]:
from pathlib import Path
import pandas as pd

ROOT_FOLDER = r"/kaggle/input/datasets/ziyadaltalhi/tik-tok-data/Tik Tok Datasets - After Processing"

root = Path(ROOT_FOLDER)

analysis_files = [
    p for p in root.rglob("*.xlsx")
    if p.name.endswith("_analysis.xlsx") and not p.name.startswith("~$")
]

print(f"Total analysis files found: {len(analysis_files)}\n")

total_rows = 0

for i, file in enumerate(sorted(analysis_files), 1):
    try:
        df = pd.read_excel(file)
        rows = len(df)
        total_rows += rows

        print(f"{i:03d} | Rows: {rows:5d} | {file.name}")

    except Exception as e:
        print(f"{i:03d} | ERROR reading file: {file.name} | {e}")

print("\n" + "="*50)
print(f"Total rows across all analysis files: {total_rows:,}")

Total analysis files found: 296

001 | Rows:    25 | Video comments 10_textready_analysis.xlsx
002 | Rows:    17 | Video comments 11_textready_analysis.xlsx
003 | Rows:    35 | Video comments 12_textready_analysis.xlsx
004 | Rows:   166 | Video comments 13_textready_analysis.xlsx
005 | Rows:    12 | Video comments 14_textready_analysis.xlsx
006 | Rows:    24 | Video comments 15_textready_analysis.xlsx
007 | Rows:    15 | Video comments 16_textready_analysis.xlsx
008 | Rows:    24 | Video comments 17_textready_analysis.xlsx
009 | Rows:    12 | Video comments 18_textready_analysis.xlsx
010 | Rows:    16 | Video comments 19_textready_analysis.xlsx
011 | Rows:    17 | Video comments 20_textready_analysis.xlsx
012 | Rows:    14 | Video comments 21_textready_analysis.xlsx
013 | Rows:    13 | Video comments 22_textready_analysis.xlsx
014 | Rows:    13 | Video comments 23_textready_analysis.xlsx
015 | Rows:    29 | Video comments 24_textready_analysis.xlsx
016 | Rows:    34 | Video comments 25

In [8]:
# =========================================
# 1) IMPORTS
# =========================================
import os
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional, SpatialDropout1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

# =========================================
# 2) PATHS
# =========================================
ROOT_FOLDER = r"/kaggle/input/datasets/ziyadaltalhi/tik-tok-data/Tik Tok Datasets - After Processing"
OUTPUT_DIR = r"/kaggle/working/tiktok_sentiment_cleaned"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================================
# 3) LOAD CLEANED FILES
# =========================================
root = Path(ROOT_FOLDER)

cleaned_files = list(root.rglob("*_cleaned.xlsx"))
print("Cleaned files:", len(cleaned_files))

dfs = []
for f in cleaned_files:
    try:
        df = pd.read_excel(f)
        dfs.append(df)
    except:
        pass

df = pd.concat(dfs, ignore_index=True)
print("Initial shape:", df.shape)

# =========================================
# 4) SELECT TEXT COLUMN
# =========================================
TEXT_COLS = ["Text_TR", "Text_ML", "Text"]

for col in TEXT_COLS:
    if col in df.columns:
        text_col = col
        break

print("Using text column:", text_col)

df = df.dropna(subset=[text_col])

# =========================================
# 5) REMOVE NOISE + ADS
# =========================================
if "comment_type" in df.columns:
    df = df[~df["comment_type"].str.lower().isin(["ad", "noise"])]

print("After removing ad/noise:", df.shape)

# =========================================
# 6) BUILD SENTIMENT LABEL
# =========================================

def build_label(row):
    # 1) لو فيه Sentiment جاهز
    if "Sentiment" in row and pd.notna(row["Sentiment"]):
        s = str(row["Sentiment"]).lower()

        if "pos" in s:
            return "positive"
        elif "neg" in s:
            return "negative"
        else:
            return "neutral"

    # 2) fallback على Stars
    if "Stars" in row and pd.notna(row["Stars"]):
        try:
            star = float(row["Stars"])
            if star >= 4:
                return "positive"
            elif star <= 2:
                return "negative"
            else:
                return "neutral"
        except:
            return None

    return None

df["label"] = df.apply(build_label, axis=1)
df = df.dropna(subset=["label"])

print("\nLabel distribution:")
print(df["label"].value_counts())

# =========================================
# 7) ENCODE
# =========================================
le = LabelEncoder()
df["y"] = le.fit_transform(df["label"])

# =========================================
# 8) SPLIT
# =========================================
X_train, X_temp, y_train, y_temp = train_test_split(
    df[text_col], df["y"], test_size=0.2, random_state=42, stratify=df["y"]
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

# =========================================
# 9) TOKENIZATION
# =========================================
MAX_WORDS = 50000
MAX_LEN = 40

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

def encode(texts):
    seq = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seq, maxlen=MAX_LEN)

X_train_pad = encode(X_train)
X_val_pad   = encode(X_val)
X_test_pad  = encode(X_test)

# =========================================
# 10) MODEL
# =========================================
model = Sequential([
    Embedding(MAX_WORDS, 128, input_length=MAX_LEN),
    SpatialDropout1D(0.3),
    Bidirectional(LSTM(128)),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(3, activation='softmax')
])

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

# =========================================
# 11) TRAIN
# =========================================
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=10,
    batch_size=128,
    callbacks=[early_stop],
    verbose=1
)

# =========================================
# 12) EVALUATE
# =========================================
y_pred = model.predict(X_test_pad)
y_pred = np.argmax(y_pred, axis=1)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

# =========================================
# 13) SAVE
# =========================================
model.save(os.path.join(OUTPUT_DIR, "sentiment_cleaned.keras"))

print("\nDone 🚀")

Cleaned files: 296
Initial shape: (113710, 76)
Using text column: Text_TR
After removing ad/noise: (61825, 76)

Label distribution:
label
neutral     30448
positive    17456
negative    13921
Name: count, dtype: int64


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_3             │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
387/387 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.6272 - loss: 0.7775 - val_accuracy: 0.8041 - val_loss: 0.4839
Epoch 2/10
387/387 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.8499 - loss: 0.3819 - val_accuracy: 0.8124 - val_loss: 0.4944
Epoch 3/10
387/387 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.9180 - loss: 0.2247 - val_accuracy: 0.8109 - val_loss: 0.5860
194/194 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step

Classification Report:

              precision    recall  f1-score   support

    negative       0.66      0.62      0.64      1392
     neutral       0.82      0.88      0.85      3045
    positive       0.83      0.78      0.80      1746

    accuracy                           0.79      6183
   macro avg       0.77      0.76      0.76      6183
weighted avg       0.79      0.79      0.79      6183


Confusion Matrix:

[[ 866  362  164]
 [ 264 2670  111]
 [ 182  206 1358]]

Done 🚀


In [9]:
# =========================================
# 1) IMPORTS
# =========================================
import os
import json
import pickle
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional, SpatialDropout1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

warnings.filterwarnings("ignore")

# =========================================
# 2) SETTINGS
# =========================================
ROOT_FOLDER = r"/kaggle/input/datasets/ziyadaltalhi/tik-tok-data/Tik Tok Datasets - After Processing"
OUTPUT_DIR = r"/kaggle/working/tiktok_sentiment_cleaned_3class_improved"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 42
MAX_WORDS = 50000
MAX_LEN = 60
EMBED_DIM = 128
LSTM_UNITS = 128
BATCH_SIZE = 128
EPOCHS = 12

# تفعيل تحسينات
USE_CLASS_WEIGHTS = True
DOWNSAMPLE_NEUTRAL = True
NEUTRAL_MAX = 18000   # جرب 15000 أو 18000 أو 20000

TEXT_COLS = ["Text_TR", "Text_ML", "Text", "Text_Orig"]

# =========================================
# 3) SEED
# =========================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(RANDOM_STATE)

# =========================================
# 4) LOAD CLEANED FILES
# =========================================
root = Path(ROOT_FOLDER)
cleaned_files = sorted(root.rglob("*_cleaned.xlsx"))

print("Cleaned files:", len(cleaned_files))

dfs = []
bad_files = []

for f in cleaned_files:
    try:
        df_part = pd.read_excel(f)
        df_part["Source_File"] = f.name
        df_part["Parent_Folder"] = f.parent.name
        dfs.append(df_part)
    except Exception as e:
        bad_files.append((str(f), str(e)))

if not dfs:
    raise ValueError("No cleaned files could be loaded.")

df = pd.concat(dfs, ignore_index=True)
print("Initial shape:", df.shape)

if bad_files:
    pd.DataFrame(bad_files, columns=["file", "error"]).to_excel(
        os.path.join(OUTPUT_DIR, "failed_cleaned_files.xlsx"), index=False
    )

# =========================================
# 5) CHOOSE TEXT COLUMN
# =========================================
text_col = None
for col in TEXT_COLS:
    if col in df.columns:
        non_empty = df[col].notna().sum()
        if non_empty > 0:
            text_col = col
            break

if text_col is None:
    raise ValueError("No suitable text column found.")

print("Using text column:", text_col)

df[text_col] = df[text_col].fillna("").astype(str).str.strip()
df = df[df[text_col] != ""].copy()

# =========================================
# 6) REMOVE ad + noise
# =========================================
if "comment_type" in df.columns:
    df["comment_type"] = df["comment_type"].astype(str).str.strip().str.lower()
    df = df[~df["comment_type"].isin(["ad", "noise"])].copy()

print("After removing ad/noise:", df.shape)

# =========================================
# 7) BUILD 3-CLASS SENTIMENT LABEL
# =========================================
def build_label(row):
    # أولًا Sentiment إذا موجود
    if "Sentiment" in row and pd.notna(row["Sentiment"]):
        s = str(row["Sentiment"]).strip().lower()
        if s in ["positive", "pos", "1"]:
            return "positive"
        elif s in ["negative", "neg", "-1"]:
            return "negative"
        elif s in ["neutral", "neu", "0"]:
            return "neutral"

    # fallback على Stars
    if "Stars" in row and pd.notna(row["Stars"]):
        try:
            star = float(row["Stars"])
            if star >= 4:
                return "positive"
            elif star <= 2:
                return "negative"
            else:
                return "neutral"
        except:
            return None

    return None

df["label"] = df.apply(build_label, axis=1)
df = df.dropna(subset=["label"]).copy()

print("\nLabel distribution before balancing:")
print(df["label"].value_counts())

# =========================================
# 8) OPTIONAL: DOWNSAMPLE NEUTRAL
# =========================================
if DOWNSAMPLE_NEUTRAL:
    neutral_df = df[df["label"] == "neutral"].copy()
    positive_df = df[df["label"] == "positive"].copy()
    negative_df = df[df["label"] == "negative"].copy()

    if len(neutral_df) > NEUTRAL_MAX:
        neutral_df = neutral_df.sample(n=NEUTRAL_MAX, random_state=RANDOM_STATE)

    df = pd.concat([positive_df, negative_df, neutral_df], ignore_index=True)
    df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print("\nLabel distribution after balancing:")
print(df["label"].value_counts())

# =========================================
# 9) ENCODE LABELS
# =========================================
le = LabelEncoder()
df["y"] = le.fit_transform(df["label"])

print("\nEncoded classes:")
for cls_name, cls_id in zip(le.classes_, le.transform(le.classes_)):
    print(f"{cls_id} -> {cls_name}")

# =========================================
# 10) SPLIT
# =========================================
X = df[text_col].astype(str).tolist()
y = df["y"].values

X_train, X_temp, y_train, y_temp, idx_train, idx_temp = train_test_split(
    X, y, df.index.values,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

X_val, X_test, y_val, y_test, idx_val, idx_test = train_test_split(
    X_temp, y_temp, idx_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

print("\nTrain size:", len(X_train))
print("Val size:", len(X_val))
print("Test size:", len(X_test))

# =========================================
# 11) TOKENIZATION
# =========================================
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

def encode_texts(texts, tokenizer, max_len):
    seq = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seq, maxlen=max_len, padding="post", truncating="post")

X_train_pad = encode_texts(X_train, tokenizer, MAX_LEN)
X_val_pad   = encode_texts(X_val, tokenizer, MAX_LEN)
X_test_pad  = encode_texts(X_test, tokenizer, MAX_LEN)

# =========================================
# 12) CLASS WEIGHTS
# =========================================
class_weight_dict = None
if USE_CLASS_WEIGHTS:
    classes = np.unique(y_train)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
    class_weight_dict = {int(c): float(w) for c, w in zip(classes, weights)}

    print("\nClass weights:")
    print(class_weight_dict)

# =========================================
# 13) MODEL
# =========================================
num_classes = len(le.classes_)

model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=EMBED_DIM),
    SpatialDropout1D(0.4),
    Bidirectional(LSTM(LSTM_UNITS, dropout=0.2, recurrent_dropout=0.2)),
    Dense(64, activation="relu"),
    Dropout(0.4),
    Dense(num_classes, activation="softmax")
])

model.build(input_shape=(None, MAX_LEN))

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model.summary()

# =========================================
# 14) CALLBACKS
# =========================================
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=1,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=os.path.join(OUTPUT_DIR, "best_3class.keras"),
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
]

# =========================================
# 15) TRAIN
# =========================================
history = model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_val_pad, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1
)

# =========================================
# 16) EVALUATE
# =========================================
y_prob = model.predict(X_test_pad, batch_size=BATCH_SIZE, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

acc = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")
f1_weighted = f1_score(y_test, y_pred, average="weighted")

print("\nAccuracy:", acc)
print("F1 Macro:", f1_macro)
print("F1 Weighted:", f1_weighted)

print("\nClassification Report:\n")
report_text = classification_report(y_test, y_pred, target_names=le.classes_)
print(report_text)

cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:\n")
print(cm)

# =========================================
# 17) SAVE REPORTS
# =========================================
report_dict = classification_report(
    y_test, y_pred,
    target_names=le.classes_,
    output_dict=True
)

pd.DataFrame(report_dict).transpose().to_excel(
    os.path.join(OUTPUT_DIR, "classification_report.xlsx")
)

pd.DataFrame(cm, index=le.classes_, columns=le.classes_).to_excel(
    os.path.join(OUTPUT_DIR, "confusion_matrix.xlsx")
)

pd.DataFrame(history.history).to_excel(
    os.path.join(OUTPUT_DIR, "training_history.xlsx"),
    index=False
)

# =========================================
# 18) SAVE TEST PREDICTIONS
# =========================================
test_df = df.loc[idx_test].copy().reset_index(drop=True)
test_df["y_true"] = y_test
test_df["y_pred"] = y_pred
test_df["true_label"] = le.inverse_transform(y_test)
test_df["pred_label"] = le.inverse_transform(y_pred)
test_df["pred_confidence"] = y_prob.max(axis=1)

for i, cls_name in enumerate(le.classes_):
    test_df[f"prob_{cls_name}"] = y_prob[:, i]

test_df.to_excel(
    os.path.join(OUTPUT_DIR, "test_predictions.xlsx"),
    index=False
)

# =========================================
# 19) SAVE MISCLASSIFIED
# =========================================
errors_df = test_df[test_df["y_true"] != test_df["y_pred"]].copy()
errors_df = errors_df.sort_values("pred_confidence", ascending=False)
errors_df.to_excel(
    os.path.join(OUTPUT_DIR, "error_analysis.xlsx"),
    index=False
)

# =========================================
# 20) SAVE MODEL + TOKENIZER + CONFIG
# =========================================
model.save(os.path.join(OUTPUT_DIR, "final_3class.keras"))

with open(os.path.join(OUTPUT_DIR, "tokenizer.pkl"), "wb") as f:
    pickle.dump(tokenizer, f)

config = {
    "ROOT_FOLDER": ROOT_FOLDER,
    "OUTPUT_DIR": OUTPUT_DIR,
    "text_col": text_col,
    "MAX_WORDS": MAX_WORDS,
    "MAX_LEN": MAX_LEN,
    "EMBED_DIM": EMBED_DIM,
    "LSTM_UNITS": LSTM_UNITS,
    "BATCH_SIZE": BATCH_SIZE,
    "EPOCHS": EPOCHS,
    "USE_CLASS_WEIGHTS": USE_CLASS_WEIGHTS,
    "DOWNSAMPLE_NEUTRAL": DOWNSAMPLE_NEUTRAL,
    "NEUTRAL_MAX": NEUTRAL_MAX,
    "classes": list(le.classes_),
    "accuracy": float(acc),
    "f1_macro": float(f1_macro),
    "f1_weighted": float(f1_weighted),
    "final_shape": list(df.shape),
    "label_distribution": {str(k): int(v) for k, v in df["label"].value_counts().to_dict().items()}
}

with open(os.path.join(OUTPUT_DIR, "config_summary.json"), "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("\nDone successfully.")
print(json.dumps({
    "accuracy": acc,
    "f1_macro": f1_macro,
    "f1_weighted": f1_weighted
}, ensure_ascii=False, indent=2))

Cleaned files: 296
Initial shape: (113710, 78)
Using text column: Text_TR
After removing ad/noise: (61825, 78)

Label distribution before balancing:
label
neutral     30448
positive    17456
negative    13921
Name: count, dtype: int64

Label distribution after balancing:
label
neutral     18000
positive    17456
negative    13921
Name: count, dtype: int64

Encoded classes:
0 -> negative
1 -> neutral
2 -> positive

Train size: 39501
Val size: 4938
Test size: 4938

Class weights:
{0: 1.1822752985543683, 1: 0.914375, 2: 0.9429246634202234}


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 60, 128)        │     6,400,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_4             │ (None, 60, 128)        │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_4 (Bidirectional) │ (None, 256)            │       263,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,679,811 (25.48 MB)

 Trainable params: 6,679,811 (25.48 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/12
309/309 ━━━━━━━━━━━━━━━━━━━━ 0s 246ms/step - accuracy: 0.5455 - loss: 0.8918
Epoch 1: val_loss improved from inf to 0.52100, saving model to /kaggle/working/tiktok_sentiment_cleaned_3class_improved/best_3class.keras
309/309 ━━━━━━━━━━━━━━━━━━━━ 85s 256ms/step - accuracy: 0.5459 - loss: 0.8913 - val_accuracy: 0.7736 - val_loss: 0.5210 - learning_rate: 0.0010
Epoch 2/12
309/309 ━━━━━━━━━━━━━━━━━━━━ 0s 246ms/step - accuracy: 0.8179 - loss: 0.4583
Epoch 2: val_loss improved from 0.52100 to 0.50805, saving model to /kaggle/working/tiktok_sentiment_cleaned_3class_improved/best_3class.keras
309/309 ━━━━━━━━━━━━━━━━━━━━ 78s 253ms/step - accuracy: 0.8179 - loss: 0.4581 - val_accuracy: 0.8009 - val_loss: 0.5080 - learning_rate: 0.0010
Epoch 3/12
309/309 ━━━━━━━━━━━━━━━━━━━━ 0s 248ms/step - accuracy: 0.8915 - loss: 0.2919
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 3: val_loss did not improve from 0.50805
309/309 ━━━━━━━━━━━━━━━━━━━━ 79s 254ms/ste

In [10]:
# =========================================
# 1) IMPORTS
# =========================================
import os
import json
import pickle
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, LSTM, Dense, Dropout, Bidirectional,
    SpatialDropout1D, GlobalMaxPooling1D
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam

warnings.filterwarnings("ignore")

# =========================================
# 2) SETTINGS
# =========================================
ROOT_FOLDER = r"/kaggle/input/datasets/ziyadaltalhi/tik-tok-data/Tik Tok Datasets - After Processing"
OUTPUT_DIR = r"/kaggle/working/tiktok_sentiment_cleaned_3class_final"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 42

# Model size (smaller = less overfitting)
MAX_WORDS = 30000
MAX_LEN = 70
EMBED_DIM = 96
LSTM_UNITS = 64

BATCH_SIZE = 128
EPOCHS = 15
LEARNING_RATE = 5e-4

USE_CLASS_WEIGHTS = True
BOOST_NEGATIVE_WEIGHT = True

DOWNSAMPLE_NEUTRAL = True
NEUTRAL_MAX = 16000

TEXT_COLS = ["Text_TR", "Text_ML", "Text", "Text_Orig"]

# =========================================
# 3) SEED
# =========================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(RANDOM_STATE)

# =========================================
# 4) LOAD CLEANED FILES
# =========================================
root = Path(ROOT_FOLDER)
cleaned_files = sorted(root.rglob("*_cleaned.xlsx"))

print("Cleaned files:", len(cleaned_files))

dfs = []
bad_files = []

for f in cleaned_files:
    try:
        df_part = pd.read_excel(f)
        df_part["Source_File"] = f.name
        df_part["Parent_Folder"] = f.parent.name
        dfs.append(df_part)
    except Exception as e:
        bad_files.append((str(f), str(e)))

if not dfs:
    raise ValueError("No cleaned files could be loaded.")

df = pd.concat(dfs, ignore_index=True)
print("Initial shape:", df.shape)

if bad_files:
    pd.DataFrame(bad_files, columns=["file", "error"]).to_excel(
        os.path.join(OUTPUT_DIR, "failed_cleaned_files.xlsx"), index=False
    )

# =========================================
# 5) CHOOSE TEXT COLUMN
# =========================================
text_col = None
for col in TEXT_COLS:
    if col in df.columns:
        non_empty = df[col].notna().sum()
        if non_empty > 0:
            text_col = col
            break

if text_col is None:
    raise ValueError("No suitable text column found.")

print("Using text column:", text_col)

df[text_col] = df[text_col].fillna("").astype(str).str.strip()
df = df[df[text_col] != ""].copy()

# remove ultra-short noisy texts
df["word_count"] = df[text_col].astype(str).str.split().str.len()
df = df[df["word_count"] >= 2].copy()

# =========================================
# 6) REMOVE ad + noise
# =========================================
if "comment_type" in df.columns:
    df["comment_type"] = df["comment_type"].astype(str).str.strip().str.lower()
    df = df[~df["comment_type"].isin(["ad", "noise"])].copy()

print("After removing ad/noise:", df.shape)

# =========================================
# 7) BUILD 3-CLASS SENTIMENT LABEL
# =========================================
def build_label(row):
    if "Sentiment" in row and pd.notna(row["Sentiment"]):
        s = str(row["Sentiment"]).strip().lower()
        if s in ["positive", "pos", "1"]:
            return "positive"
        elif s in ["negative", "neg", "-1"]:
            return "negative"
        elif s in ["neutral", "neu", "0"]:
            return "neutral"

    if "Stars" in row and pd.notna(row["Stars"]):
        try:
            star = float(row["Stars"])
            if star >= 4:
                return "positive"
            elif star <= 2:
                return "negative"
            else:
                return "neutral"
        except:
            return None

    return None

df["label"] = df.apply(build_label, axis=1)
df = df.dropna(subset=["label"]).copy()

print("\nLabel distribution before balancing:")
print(df["label"].value_counts())

# =========================================
# 8) DOWNSAMPLE NEUTRAL
# =========================================
if DOWNSAMPLE_NEUTRAL:
    neutral_df = df[df["label"] == "neutral"].copy()
    positive_df = df[df["label"] == "positive"].copy()
    negative_df = df[df["label"] == "negative"].copy()

    if len(neutral_df) > NEUTRAL_MAX:
        neutral_df = neutral_df.sample(n=NEUTRAL_MAX, random_state=RANDOM_STATE)

    df = pd.concat([positive_df, negative_df, neutral_df], ignore_index=True)
    df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print("\nLabel distribution after balancing:")
print(df["label"].value_counts())

# =========================================
# 9) ENCODE LABELS
# =========================================
le = LabelEncoder()
df["y"] = le.fit_transform(df["label"])

print("\nEncoded classes:")
for cls_name, cls_id in zip(le.classes_, le.transform(le.classes_)):
    print(f"{cls_id} -> {cls_name}")

# =========================================
# 10) SPLIT
# =========================================
X = df[text_col].astype(str).tolist()
y = df["y"].values

X_train, X_temp, y_train, y_temp, idx_train, idx_temp = train_test_split(
    X, y, df.index.values,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

X_val, X_test, y_val, y_test, idx_val, idx_test = train_test_split(
    X_temp, y_temp, idx_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

print("\nTrain size:", len(X_train))
print("Val size:", len(X_val))
print("Test size:", len(X_test))

# =========================================
# 11) TOKENIZATION
# =========================================
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

def encode_texts(texts, tokenizer, max_len):
    seq = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seq, maxlen=max_len, padding="post", truncating="post")

X_train_pad = encode_texts(X_train, tokenizer, MAX_LEN)
X_val_pad   = encode_texts(X_val, tokenizer, MAX_LEN)
X_test_pad  = encode_texts(X_test, tokenizer, MAX_LEN)

# =========================================
# 12) CLASS WEIGHTS
# =========================================
class_weight_dict = None

if USE_CLASS_WEIGHTS:
    classes = np.unique(y_train)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
    class_weight_dict = {int(c): float(w) for c, w in zip(classes, weights)}

    # boost negative slightly
    if BOOST_NEGATIVE_WEIGHT:
        negative_id = int(le.transform(["negative"])[0])
        class_weight_dict[negative_id] *= 1.15

    print("\nClass weights:")
    print(class_weight_dict)

# =========================================
# 13) MODEL
# =========================================
num_classes = len(le.classes_)

model = Sequential([
    Embedding(
        input_dim=MAX_WORDS,
        output_dim=EMBED_DIM,
        input_length=MAX_LEN
    ),
    SpatialDropout1D(0.35),

    Bidirectional(
        LSTM(
            LSTM_UNITS,
            return_sequences=True,
            dropout=0.25,
            recurrent_dropout=0.25,
            kernel_regularizer=l2(1e-4),
            recurrent_regularizer=l2(1e-4)
        )
    ),

    GlobalMaxPooling1D(),

    Dense(64, activation="relu", kernel_regularizer=l2(1e-4)),
    Dropout(0.50),

    Dense(num_classes, activation="softmax")
])

optimizer = Adam(learning_rate=LEARNING_RATE, clipnorm=1.0)

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=optimizer,
    metrics=["accuracy"]
)

model.summary()

# =========================================
# 14) CALLBACKS
# =========================================
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=1,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=os.path.join(OUTPUT_DIR, "best_3class.keras"),
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
]

# =========================================
# 15) TRAIN
# =========================================
history = model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_val_pad, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1
)

# =========================================
# 16) EVALUATE
# =========================================
y_prob = model.predict(X_test_pad, batch_size=BATCH_SIZE, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

acc = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")
f1_weighted = f1_score(y_test, y_pred, average="weighted")

print("\nAccuracy:", acc)
print("F1 Macro:", f1_macro)
print("F1 Weighted:", f1_weighted)

print("\nClassification Report:\n")
report_text = classification_report(y_test, y_pred, target_names=le.classes_)
print(report_text)

cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:\n")
print(cm)

# =========================================
# 17) SAVE REPORTS
# =========================================
report_dict = classification_report(
    y_test, y_pred,
    target_names=le.classes_,
    output_dict=True
)

pd.DataFrame(report_dict).transpose().to_excel(
    os.path.join(OUTPUT_DIR, "classification_report.xlsx")
)

pd.DataFrame(cm, index=le.classes_, columns=le.classes_).to_excel(
    os.path.join(OUTPUT_DIR, "confusion_matrix.xlsx")
)

pd.DataFrame(history.history).to_excel(
    os.path.join(OUTPUT_DIR, "training_history.xlsx"),
    index=False
)

# =========================================
# 18) SAVE TEST PREDICTIONS
# =========================================
test_df = df.loc[idx_test].copy().reset_index(drop=True)
test_df["y_true"] = y_test
test_df["y_pred"] = y_pred
test_df["true_label"] = le.inverse_transform(y_test)
test_df["pred_label"] = le.inverse_transform(y_pred)
test_df["pred_confidence"] = y_prob.max(axis=1)

for i, cls_name in enumerate(le.classes_):
    test_df[f"prob_{cls_name}"] = y_prob[:, i]

test_df.to_excel(
    os.path.join(OUTPUT_DIR, "test_predictions.xlsx"),
    index=False
)

# =========================================
# 19) SAVE MISCLASSIFIED
# =========================================
errors_df = test_df[test_df["y_true"] != test_df["y_pred"]].copy()
errors_df = errors_df.sort_values("pred_confidence", ascending=False)
errors_df.to_excel(
    os.path.join(OUTPUT_DIR, "error_analysis.xlsx"),
    index=False
)

# =========================================
# 20) SAVE MODEL + TOKENIZER + CONFIG
# =========================================
model.save(os.path.join(OUTPUT_DIR, "final_3class.keras"))

with open(os.path.join(OUTPUT_DIR, "tokenizer.pkl"), "wb") as f:
    pickle.dump(tokenizer, f)

config = {
    "ROOT_FOLDER": ROOT_FOLDER,
    "OUTPUT_DIR": OUTPUT_DIR,
    "text_col": text_col,
    "MAX_WORDS": MAX_WORDS,
    "MAX_LEN": MAX_LEN,
    "EMBED_DIM": EMBED_DIM,
    "LSTM_UNITS": LSTM_UNITS,
    "BATCH_SIZE": BATCH_SIZE,
    "EPOCHS": EPOCHS,
    "LEARNING_RATE": LEARNING_RATE,
    "USE_CLASS_WEIGHTS": USE_CLASS_WEIGHTS,
    "BOOST_NEGATIVE_WEIGHT": BOOST_NEGATIVE_WEIGHT,
    "DOWNSAMPLE_NEUTRAL": DOWNSAMPLE_NEUTRAL,
    "NEUTRAL_MAX": NEUTRAL_MAX,
    "classes": list(le.classes_),
    "accuracy": float(acc),
    "f1_macro": float(f1_macro),
    "f1_weighted": float(f1_weighted),
    "final_shape": list(df.shape),
    "label_distribution": {str(k): int(v) for k, v in df["label"].value_counts().to_dict().items()}
}

with open(os.path.join(OUTPUT_DIR, "config_summary.json"), "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("\nDone successfully.")
print(json.dumps({
    "accuracy": acc,
    "f1_macro": f1_macro,
    "f1_weighted": f1_weighted
}, ensure_ascii=False, indent=2))

Cleaned files: 296
Initial shape: (113710, 78)
Using text column: Text_TR
After removing ad/noise: (61774, 79)

Label distribution before balancing:
label
neutral     30420
positive    17443
negative    13911
Name: count, dtype: int64

Label distribution after balancing:
label
positive    17443
neutral     16000
negative    13911
Name: count, dtype: int64

Encoded classes:
0 -> negative
1 -> neutral
2 -> positive

Train size: 37883
Val size: 4735
Test size: 4736

Class weights:
{0: 1.3048626710995297, 1: 0.9865364583333334, 2: 0.9049495962925804}


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_5             │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_5 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
296/296 ━━━━━━━━━━━━━━━━━━━━ 0s 289ms/step - accuracy: 0.3869 - loss: 1.1200
Epoch 1: val_loss improved from inf to 0.63165, saving model to /kaggle/working/tiktok_sentiment_cleaned_3class_final/best_3class.keras
296/296 ━━━━━━━━━━━━━━━━━━━━ 95s 300ms/step - accuracy: 0.3873 - loss: 1.1195 - val_accuracy: 0.7388 - val_loss: 0.6317 - learning_rate: 5.0000e-04
Epoch 2/15
296/296 ━━━━━━━━━━━━━━━━━━━━ 0s 289ms/step - accuracy: 0.7656 - loss: 0.6209
Epoch 2: val_loss improved from 0.63165 to 0.54324, saving model to /kaggle/working/tiktok_sentiment_cleaned_3class_final/best_3class.keras
296/296 ━━━━━━━━━━━━━━━━━━━━ 88s 297ms/step - accuracy: 0.7657 - loss: 0.6207 - val_accuracy: 0.7890 - val_loss: 0.5432 - learning_rate: 5.0000e-04
Epoch 3/15
296/296 ━━━━━━━━━━━━━━━━━━━━ 0s 289ms/step - accuracy: 0.8488 - loss: 0.4412
Epoch 3: val_loss improved from 0.54324 to 0.53369, saving model to /kaggle/working/tiktok_sentiment_cleaned_3class_final/best_3class.keras
296/296 ━━━━━━━━━━━━━━━━

In [11]:
# =========================================
# SAVE + LOAD + PREDICT (ALL-IN-ONE)
# =========================================

import os
import json
import pickle
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

# =========================================
# 1) SAVE EVERYTHING
# =========================================

SAVE_DIR = "/kaggle/working/final_model_package"
os.makedirs(SAVE_DIR, exist_ok=True)

# حفظ المودل
model.save(os.path.join(SAVE_DIR, "model.keras"))

# حفظ التوكنایزر
with open(os.path.join(SAVE_DIR, "tokenizer.pkl"), "wb") as f:
    pickle.dump(tokenizer, f)

# حفظ اللابل انكودر
with open(os.path.join(SAVE_DIR, "label_encoder.pkl"), "wb") as f:
    pickle.dump(le, f)

# حفظ الإعدادات
config = {
    "MAX_LEN": MAX_LEN,
    "MAX_WORDS": MAX_WORDS,
    "classes": list(le.classes_)
}

with open(os.path.join(SAVE_DIR, "config.json"), "w") as f:
    json.dump(config, f)

print("Model saved successfully at:", SAVE_DIR)


# =========================================
# 2) LOAD EVERYTHING
# =========================================

# تحميل المودل
model_loaded = load_model(os.path.join(SAVE_DIR, "model.keras"))

# تحميل التوكنایزر
with open(os.path.join(SAVE_DIR, "tokenizer.pkl"), "rb") as f:
    tokenizer_loaded = pickle.load(f)

# تحميل اللابل انكودر
with open(os.path.join(SAVE_DIR, "label_encoder.pkl"), "rb") as f:
    le_loaded = pickle.load(f)

# تحميل الإعدادات
with open(os.path.join(SAVE_DIR, "config.json"), "r") as f:
    config_loaded = json.load(f)

MAX_LEN_LOADED = config_loaded["MAX_LEN"]


# =========================================
# 3) PREDICTION FUNCTION
# =========================================

def predict_text(text):
    seq = tokenizer_loaded.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=MAX_LEN_LOADED, padding="post")

    pred = model_loaded.predict(padded, verbose=0)
    label_index = np.argmax(pred)

    label = le_loaded.inverse_transform([label_index])[0]
    confidence = float(np.max(pred))

    return label, confidence


# =========================================
# 4) TEST
# =========================================

sample_texts = [
    "المكان جميل جدا",
    "سيء جدا وما يستاهل",
    "مكان عادي"
]

for t in sample_texts:
    label, conf = predict_text(t)
    print(f"{t} --> {label} ({conf:.2f})")

Model saved successfully at: /kaggle/working/final_model_package
المكان جميل جدا --> positive (0.99)
سيء جدا وما يستاهل --> negative (0.99)
مكان عادي --> negative (0.56)


In [1]:
# =========================================
# 1) IMPORTS
# =========================================
import os
import json
import pickle
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, LSTM, Dense, Dropout, Bidirectional,
    SpatialDropout1D, GlobalMaxPooling1D
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam

warnings.filterwarnings("ignore")

# =========================================
# 2) SETTINGS
# =========================================
ROOT_FOLDER = r"/kaggle/input/datasets/ziyadaltalhi/tik-tok-data/Tik Tok Datasets - After Processing"
OUTPUT_DIR = r"/kaggle/working/tiktok_sentiment_cleaned_3class_tuned_v2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 42

MAX_WORDS = 30000
MAX_LEN = 80
EMBED_DIM = 96

BATCH_SIZE = 96
EPOCHS = 15
LEARNING_RATE = 3e-4

USE_CLASS_WEIGHTS = True
BOOST_NEGATIVE_WEIGHT = True
NEGATIVE_BOOST_FACTOR = 1.08

DOWNSAMPLE_NEUTRAL = True
NEUTRAL_MAX = 15000

TEXT_COLS = ["Text_TR", "Text_ML", "Text", "Text_Orig"]

# =========================================
# 3) SEED
# =========================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(RANDOM_STATE)

# =========================================
# 4) LOAD CLEANED FILES
# =========================================
root = Path(ROOT_FOLDER)
cleaned_files = sorted(root.rglob("*_cleaned.xlsx"))

print("Cleaned files:", len(cleaned_files))

dfs = []
bad_files = []

for f in cleaned_files:
    try:
        df_part = pd.read_excel(f)
        df_part["Source_File"] = f.name
        df_part["Parent_Folder"] = f.parent.name
        dfs.append(df_part)
    except Exception as e:
        bad_files.append((str(f), str(e)))

if not dfs:
    raise ValueError("No cleaned files could be loaded.")

df = pd.concat(dfs, ignore_index=True)
print("Initial shape:", df.shape)

if bad_files:
    pd.DataFrame(bad_files, columns=["file", "error"]).to_excel(
        os.path.join(OUTPUT_DIR, "failed_cleaned_files.xlsx"), index=False
    )

# =========================================
# 5) CHOOSE TEXT COLUMN
# =========================================
text_col = None
for col in TEXT_COLS:
    if col in df.columns:
        non_empty = df[col].notna().sum()
        if non_empty > 0:
            text_col = col
            break

if text_col is None:
    raise ValueError("No suitable text column found.")

print("Using text column:", text_col)

df[text_col] = df[text_col].fillna("").astype(str).str.strip()
df = df[df[text_col] != ""].copy()

df["word_count"] = df[text_col].astype(str).str.split().str.len()
df = df[df["word_count"] >= 2].copy()

# =========================================
# 6) REMOVE ad + noise
# =========================================
if "comment_type" in df.columns:
    df["comment_type"] = df["comment_type"].astype(str).str.strip().str.lower()
    df = df[~df["comment_type"].isin(["ad", "noise"])].copy()

print("After removing ad/noise:", df.shape)

# =========================================
# 7) BUILD LABEL
# =========================================
def build_label(row):
    if "Sentiment" in row and pd.notna(row["Sentiment"]):
        s = str(row["Sentiment"]).strip().lower()
        if s in ["positive", "pos", "1"]:
            return "positive"
        elif s in ["negative", "neg", "-1"]:
            return "negative"
        elif s in ["neutral", "neu", "0"]:
            return "neutral"

    if "Stars" in row and pd.notna(row["Stars"]):
        try:
            star = float(row["Stars"])
            if star >= 4:
                return "positive"
            elif star <= 2:
                return "negative"
            else:
                return "neutral"
        except:
            return None

    return None

df["label"] = df.apply(build_label, axis=1)
df = df.dropna(subset=["label"]).copy()

print("\nLabel distribution before balancing:")
print(df["label"].value_counts())

# =========================================
# 8) DOWNSAMPLE NEUTRAL
# =========================================
if DOWNSAMPLE_NEUTRAL:
    neutral_df = df[df["label"] == "neutral"].copy()
    positive_df = df[df["label"] == "positive"].copy()
    negative_df = df[df["label"] == "negative"].copy()

    if len(neutral_df) > NEUTRAL_MAX:
        neutral_df = neutral_df.sample(n=NEUTRAL_MAX, random_state=RANDOM_STATE)

    df = pd.concat([positive_df, negative_df, neutral_df], ignore_index=True)
    df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print("\nLabel distribution after balancing:")
print(df["label"].value_counts())

# =========================================
# 9) ENCODE
# =========================================
le = LabelEncoder()
df["y"] = le.fit_transform(df["label"])

print("\nEncoded classes:")
for cls_name, cls_id in zip(le.classes_, le.transform(le.classes_)):
    print(f"{cls_id} -> {cls_name}")

# =========================================
# 10) SPLIT
# =========================================
X = df[text_col].astype(str).tolist()
y = df["y"].values

X_train, X_temp, y_train, y_temp, idx_train, idx_temp = train_test_split(
    X, y, df.index.values,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

X_val, X_test, y_val, y_test, idx_val, idx_test = train_test_split(
    X_temp, y_temp, idx_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

print("\nTrain size:", len(X_train))
print("Val size:", len(X_val))
print("Test size:", len(X_test))

# =========================================
# 11) TOKENIZATION
# =========================================
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

def encode_texts(texts, tokenizer, max_len):
    seq = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seq, maxlen=max_len, padding="post", truncating="post")

X_train_pad = encode_texts(X_train, tokenizer, MAX_LEN)
X_val_pad   = encode_texts(X_val, tokenizer, MAX_LEN)
X_test_pad  = encode_texts(X_test, tokenizer, MAX_LEN)

# =========================================
# 12) CLASS WEIGHTS
# =========================================
class_weight_dict = None

if USE_CLASS_WEIGHTS:
    classes = np.unique(y_train)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
    class_weight_dict = {int(c): float(w) for c, w in zip(classes, weights)}

    if BOOST_NEGATIVE_WEIGHT:
        negative_id = int(le.transform(["negative"])[0])
        class_weight_dict[negative_id] *= NEGATIVE_BOOST_FACTOR

    print("\nClass weights:")
    print(class_weight_dict)

# =========================================
# 13) MODEL
# =========================================
num_classes = len(le.classes_)

model = Sequential([
    Embedding(
        input_dim=MAX_WORDS,
        output_dim=EMBED_DIM,
        input_length=MAX_LEN
    ),
    SpatialDropout1D(0.30),

    Bidirectional(
        LSTM(
            48,
            return_sequences=True,
            dropout=0.20,
            recurrent_dropout=0.20,
            kernel_regularizer=l2(8e-5),
            recurrent_regularizer=l2(8e-5)
        )
    ),

    Bidirectional(
        LSTM(
            32,
            return_sequences=True,
            dropout=0.20,
            recurrent_dropout=0.20,
            kernel_regularizer=l2(8e-5),
            recurrent_regularizer=l2(8e-5)
        )
    ),

    GlobalMaxPooling1D(),

    Dense(48, activation="relu", kernel_regularizer=l2(8e-5)),
    Dropout(0.45),

    Dense(num_classes, activation="softmax")
])

optimizer = Adam(learning_rate=LEARNING_RATE, clipnorm=1.0)

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=optimizer,
    metrics=["accuracy"]
)

model.summary()

# =========================================
# 14) CALLBACKS
# =========================================
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=1,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=os.path.join(OUTPUT_DIR, "best_3class.keras"),
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
]

# =========================================
# 15) TRAIN
# =========================================
history = model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_val_pad, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1
)

# =========================================
# 16) EVALUATE
# =========================================
y_prob = model.predict(X_test_pad, batch_size=BATCH_SIZE, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

acc = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")
f1_weighted = f1_score(y_test, y_pred, average="weighted")

print("\nAccuracy:", acc)
print("F1 Macro:", f1_macro)
print("F1 Weighted:", f1_weighted)

print("\nClassification Report:\n")
report_text = classification_report(y_test, y_pred, target_names=le.classes_)
print(report_text)

cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:\n")
print(cm)

# =========================================
# 17) SAVE OUTPUTS
# =========================================
report_dict = classification_report(
    y_test, y_pred,
    target_names=le.classes_,
    output_dict=True
)

pd.DataFrame(report_dict).transpose().to_excel(
    os.path.join(OUTPUT_DIR, "classification_report.xlsx")
)

pd.DataFrame(cm, index=le.classes_, columns=le.classes_).to_excel(
    os.path.join(OUTPUT_DIR, "confusion_matrix.xlsx")
)

pd.DataFrame(history.history).to_excel(
    os.path.join(OUTPUT_DIR, "training_history.xlsx"),
    index=False
)

test_df = df.loc[idx_test].copy().reset_index(drop=True)
test_df["y_true"] = y_test
test_df["y_pred"] = y_pred
test_df["true_label"] = le.inverse_transform(y_test)
test_df["pred_label"] = le.inverse_transform(y_pred)
test_df["pred_confidence"] = y_prob.max(axis=1)

for i, cls_name in enumerate(le.classes_):
    test_df[f"prob_{cls_name}"] = y_prob[:, i]

test_df.to_excel(
    os.path.join(OUTPUT_DIR, "test_predictions.xlsx"),
    index=False
)

errors_df = test_df[test_df["y_true"] != test_df["y_pred"]].copy()
errors_df = errors_df.sort_values("pred_confidence", ascending=False)
errors_df.to_excel(
    os.path.join(OUTPUT_DIR, "error_analysis.xlsx"),
    index=False
)

model.save(os.path.join(OUTPUT_DIR, "final_3class.keras"))

with open(os.path.join(OUTPUT_DIR, "tokenizer.pkl"), "wb") as f:
    pickle.dump(tokenizer, f)

config = {
    "ROOT_FOLDER": ROOT_FOLDER,
    "OUTPUT_DIR": OUTPUT_DIR,
    "text_col": text_col,
    "MAX_WORDS": MAX_WORDS,
    "MAX_LEN": MAX_LEN,
    "EMBED_DIM": EMBED_DIM,
    "BATCH_SIZE": BATCH_SIZE,
    "EPOCHS": EPOCHS,
    "LEARNING_RATE": LEARNING_RATE,
    "USE_CLASS_WEIGHTS": USE_CLASS_WEIGHTS,
    "BOOST_NEGATIVE_WEIGHT": BOOST_NEGATIVE_WEIGHT,
    "NEGATIVE_BOOST_FACTOR": NEGATIVE_BOOST_FACTOR,
    "DOWNSAMPLE_NEUTRAL": DOWNSAMPLE_NEUTRAL,
    "NEUTRAL_MAX": NEUTRAL_MAX,
    "classes": list(le.classes_),
    "accuracy": float(acc),
    "f1_macro": float(f1_macro),
    "f1_weighted": float(f1_weighted),
    "final_shape": list(df.shape),
    "label_distribution": {str(k): int(v) for k, v in df["label"].value_counts().to_dict().items()}
}

with open(os.path.join(OUTPUT_DIR, "config_summary.json"), "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("\nDone successfully.")
print(json.dumps({
    "accuracy": acc,
    "f1_macro": f1_macro,
    "f1_weighted": f1_weighted
}, ensure_ascii=False, indent=2))

2026-04-14 18:13:37.024065: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776190417.182979      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776190417.227453      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776190417.610228      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776190417.610274      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776190417.610279      55 computation_placer.cc:177] computation placer alr

Cleaned files: 296
Initial shape: (113710, 78)
Using text column: Text_TR
After removing ad/noise: (61774, 79)

Label distribution before balancing:
label
neutral     30420
positive    17443
negative    13911
Name: count, dtype: int64

Label distribution after balancing:
label
positive    17443
neutral     15000
negative    13911
Name: count, dtype: int64

Encoded classes:
0 -> negative
1 -> neutral
2 -> positive

Train size: 37083
Val size: 4635
Test size: 4636

Class weights:
{0: 1.1995579117620632, 1: 1.0300833333333332, 2: 0.8858391858965171}


I0000 00:00:1776190496.077355      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1776190496.083312      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
387/387 ━━━━━━━━━━━━━━━━━━━━ 0s 730ms/step - accuracy: 0.3545 - loss: 1.1401
Epoch 1: val_loss improved from inf to 0.72327, saving model to /kaggle/working/tiktok_sentiment_cleaned_3class_tuned_v2/best_3class.keras
387/387 ━━━━━━━━━━━━━━━━━━━━ 307s 751ms/step - accuracy: 0.3548 - loss: 1.1398 - val_accuracy: 0.6785 - val_loss: 0.7233 - learning_rate: 3.0000e-04
Epoch 2/15
387/387 ━━━━━━━━━━━━━━━━━━━━ 0s 734ms/step - accuracy: 0.7093 - loss: 0.7129
Epoch 2: val_loss improved from 0.72327 to 0.60617, saving model to /kaggle/working/tiktok_sentiment_cleaned_3class_tuned_v2/best_3class.keras
387/387 ━━━━━━━━━━━━━━━━━━━━ 291s 751ms/step - accuracy: 0.7094 - loss: 0.7128 - val_accuracy: 0.7484 - val_loss: 0.6062 - learning_rate: 3.0000e-04
Epoch 3/15
387/387 ━━━━━━━━━━━━━━━━━━━━ 0s 734ms/step - accuracy: 0.8064 - loss: 0.5333
Epoch 3: val_loss improved from 0.60617 to 0.57163, saving model to /kaggle/working/tiktok_sentiment_cleaned_3class_tuned_v2/best_3class.keras
387/387 ━━━━━

In [1]:
# =========================================
# 1) IMPORTS
# =========================================
import os
import json
import pickle
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, LSTM, Dense, Dropout, Bidirectional,
    SpatialDropout1D, GlobalMaxPooling1D
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam

warnings.filterwarnings("ignore")

# =========================================
# 2) SETTINGS
# =========================================
ROOT_FOLDER = r"/kaggle/input/datasets/ziyadaltalhi/tik-tok-data/Tik Tok Datasets - After Processing"
OUTPUT_DIR = r"/kaggle/working/tiktok_sentiment_binary"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 42

TEXT_COL = "Text_TR"

MAX_WORDS = 30000
MAX_LEN = 80
EMBED_DIM = 96

BATCH_SIZE = 96
EPOCHS = 15
LEARNING_RATE = 3e-4

USE_CLASS_WEIGHTS = True
BOOST_NEGATIVE_WEIGHT = True
NEGATIVE_BOOST_FACTOR = 1.10

# نحذف الفارغ فقط، ونبقي النصوص القصيرة غير الفارغة
REMOVE_EMPTY_ONLY = True

EXCLUDED_COMMENT_TYPES = {
    "ad",
    "ads",
    "noise",
    "noisy",
    "question",
    "questions",
    "channel_comment"
}

# =========================================
# 3) SEED
# =========================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(RANDOM_STATE)

# =========================================
# 4) LOAD CLEANED FILES
# =========================================
root = Path(ROOT_FOLDER)

# cleaned only
cleaned_files = sorted(root.rglob("*_cleaned.xlsx"))

print("Cleaned files:", len(cleaned_files))

dfs = []
bad_files = []

for f in cleaned_files:
    try:
        df_part = pd.read_excel(f)
        df_part["Source_File"] = f.name
        df_part["Parent_Folder"] = f.parent.name
        dfs.append(df_part)
    except Exception as e:
        bad_files.append((str(f), str(e)))

if not dfs:
    raise ValueError("No cleaned files could be loaded.")

df = pd.concat(dfs, ignore_index=True)
print("Initial shape:", df.shape)

if bad_files:
    pd.DataFrame(bad_files, columns=["file", "error"]).to_excel(
        os.path.join(OUTPUT_DIR, "failed_cleaned_files.xlsx"), index=False
    )

# =========================================
# 5) KEEP ONLY Text_TR
# =========================================
if TEXT_COL not in df.columns:
    raise ValueError(f"{TEXT_COL} column was not found.")

df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str).str.strip()
df[TEXT_COL] = df[TEXT_COL].str.replace(r"\s+", " ", regex=True)

before_text = len(df)
if REMOVE_EMPTY_ONLY:
    df = df[df[TEXT_COL] != ""].copy()
after_text = len(df)

print(f"After Text_TR non-empty filtering: {before_text} -> {after_text}")

# =========================================
# 6) REMOVE UNWANTED COMMENT TYPES
# =========================================
if "comment_type" in df.columns:
    df["comment_type"] = df["comment_type"].fillna("").astype(str).str.strip().str.lower()
    df = df[~df["comment_type"].isin(EXCLUDED_COMMENT_TYPES)].copy()

print("After removing excluded comment types:", df.shape)

# =========================================
# 7) BUILD BINARY LABEL FROM Stars ONLY
# positive / negative فقط
# =========================================
def build_binary_label(row):
    if "Stars" in row and pd.notna(row["Stars"]):
        try:
            star = float(row["Stars"])
            if star >= 4:
                return "positive"
            elif star <= 2:
                return "negative"
            else:
                return None   # حذف المحايد
        except:
            return None
    return None

df["label"] = df.apply(build_binary_label, axis=1)
df = df.dropna(subset=["label"]).copy()

print("\nBinary label distribution:")
print(df["label"].value_counts())

# =========================================
# 8) ENCODE
# =========================================
le = LabelEncoder()
df["y"] = le.fit_transform(df["label"])

print("\nEncoded classes:")
for cls_name, cls_id in zip(le.classes_, le.transform(le.classes_)):
    print(f"{cls_id} -> {cls_name}")

# =========================================
# 9) SPLIT
# =========================================
X = df[TEXT_COL].astype(str).tolist()
y = df["y"].values

X_train, X_temp, y_train, y_temp, idx_train, idx_temp = train_test_split(
    X, y, df.index.values,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

X_val, X_test, y_val, y_test, idx_val, idx_test = train_test_split(
    X_temp, y_temp, idx_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

print("\nTrain size:", len(X_train))
print("Val size:", len(X_val))
print("Test size:", len(X_test))

# =========================================
# 10) TOKENIZATION
# =========================================
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

def encode_texts(texts, tokenizer, max_len):
    seq = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seq, maxlen=max_len, padding="post", truncating="post")

X_train_pad = encode_texts(X_train, tokenizer, MAX_LEN)
X_val_pad   = encode_texts(X_val, tokenizer, MAX_LEN)
X_test_pad  = encode_texts(X_test, tokenizer, MAX_LEN)

# =========================================
# 11) CLASS WEIGHTS
# =========================================
class_weight_dict = None

if USE_CLASS_WEIGHTS:
    classes = np.unique(y_train)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
    class_weight_dict = {int(c): float(w) for c, w in zip(classes, weights)}

    if BOOST_NEGATIVE_WEIGHT and "negative" in le.classes_:
        negative_id = int(le.transform(["negative"])[0])
        class_weight_dict[negative_id] *= NEGATIVE_BOOST_FACTOR

    print("\nClass weights:")
    print(class_weight_dict)

# =========================================
# 12) MODEL
# =========================================
num_classes = len(le.classes_)

model = Sequential([
    Embedding(
        input_dim=MAX_WORDS,
        output_dim=EMBED_DIM,
        input_length=MAX_LEN
    ),
    SpatialDropout1D(0.30),

    Bidirectional(
        LSTM(
            48,
            return_sequences=True,
            dropout=0.20,
            recurrent_dropout=0.20,
            kernel_regularizer=l2(8e-5),
            recurrent_regularizer=l2(8e-5)
        )
    ),

    Bidirectional(
        LSTM(
            32,
            return_sequences=True,
            dropout=0.20,
            recurrent_dropout=0.20,
            kernel_regularizer=l2(8e-5),
            recurrent_regularizer=l2(8e-5)
        )
    ),

    GlobalMaxPooling1D(),

    Dense(48, activation="relu", kernel_regularizer=l2(8e-5)),
    Dropout(0.45),

    Dense(num_classes, activation="softmax")
])

optimizer = Adam(learning_rate=LEARNING_RATE, clipnorm=1.0)

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=optimizer,
    metrics=["accuracy"]
)

model.summary()

# =========================================
# 13) CALLBACKS
# =========================================
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=1,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=os.path.join(OUTPUT_DIR, "best_binary.keras"),
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
]

# =========================================
# 14) TRAIN
# =========================================
history = model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_val_pad, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1
)

# =========================================
# 15) EVALUATE
# =========================================
y_prob = model.predict(X_test_pad, batch_size=BATCH_SIZE, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

acc = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")
f1_weighted = f1_score(y_test, y_pred, average="weighted")

print("\nAccuracy:", acc)
print("F1 Macro:", f1_macro)
print("F1 Weighted:", f1_weighted)

print("\nClassification Report:\n")
report_text = classification_report(y_test, y_pred, target_names=le.classes_)
print(report_text)

cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:\n")
print(cm)

# =========================================
# 16) SAVE OUTPUTS
# =========================================
report_dict = classification_report(
    y_test, y_pred,
    target_names=le.classes_,
    output_dict=True
)

pd.DataFrame(report_dict).transpose().to_excel(
    os.path.join(OUTPUT_DIR, "classification_report.xlsx")
)

pd.DataFrame(cm, index=le.classes_, columns=le.classes_).to_excel(
    os.path.join(OUTPUT_DIR, "confusion_matrix.xlsx")
)

pd.DataFrame(history.history).to_excel(
    os.path.join(OUTPUT_DIR, "training_history.xlsx"),
    index=False
)

test_df = df.loc[idx_test].copy().reset_index(drop=True)
test_df["y_true"] = y_test
test_df["y_pred"] = y_pred
test_df["true_label"] = le.inverse_transform(y_test)
test_df["pred_label"] = le.inverse_transform(y_pred)
test_df["pred_confidence"] = y_prob.max(axis=1)

for i, cls_name in enumerate(le.classes_):
    test_df[f"prob_{cls_name}"] = y_prob[:, i]

test_df.to_excel(
    os.path.join(OUTPUT_DIR, "test_predictions.xlsx"),
    index=False
)

errors_df = test_df[test_df["y_true"] != test_df["y_pred"]].copy()
errors_df = errors_df.sort_values("pred_confidence", ascending=False)
errors_df.to_excel(
    os.path.join(OUTPUT_DIR, "error_analysis.xlsx"),
    index=False
)

model.save(os.path.join(OUTPUT_DIR, "final_binary.keras"))

with open(os.path.join(OUTPUT_DIR, "tokenizer.pkl"), "wb") as f:
    pickle.dump(tokenizer, f)

with open(os.path.join(OUTPUT_DIR, "label_encoder.pkl"), "wb") as f:
    pickle.dump(le, f)

config = {
    "ROOT_FOLDER": ROOT_FOLDER,
    "OUTPUT_DIR": OUTPUT_DIR,
    "TEXT_COL": TEXT_COL,
    "MAX_WORDS": MAX_WORDS,
    "MAX_LEN": MAX_LEN,
    "EMBED_DIM": EMBED_DIM,
    "BATCH_SIZE": BATCH_SIZE,
    "EPOCHS": EPOCHS,
    "LEARNING_RATE": LEARNING_RATE,
    "USE_CLASS_WEIGHTS": USE_CLASS_WEIGHTS,
    "BOOST_NEGATIVE_WEIGHT": BOOST_NEGATIVE_WEIGHT,
    "NEGATIVE_BOOST_FACTOR": NEGATIVE_BOOST_FACTOR,
    "REMOVE_EMPTY_ONLY": REMOVE_EMPTY_ONLY,
    "EXCLUDED_COMMENT_TYPES": sorted(list(EXCLUDED_COMMENT_TYPES)),
    "classes": list(le.classes_),
    "accuracy": float(acc),
    "f1_macro": float(f1_macro),
    "f1_weighted": float(f1_weighted),
    "final_shape": list(df.shape),
    "label_distribution": {str(k): int(v) for k, v in df["label"].value_counts().to_dict().items()}
}

with open(os.path.join(OUTPUT_DIR, "config_summary.json"), "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("\nDone successfully.")
print(json.dumps({
    "accuracy": acc,
    "f1_macro": f1_macro,
    "f1_weighted": f1_weighted
}, ensure_ascii=False, indent=2))

2026-04-18 12:32:55.047760: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776515575.430903      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776515575.544640      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776515576.525224      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776515576.525266      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776515576.525268      55 computation_placer.cc:177] computation placer alr

Cleaned files: 296
Initial shape: (113710, 78)
After Text_TR non-empty filtering: 113710 -> 113710
After removing excluded comment types: (44189, 78)

Binary label distribution:
label
positive    17385
negative    13758
Name: count, dtype: int64

Encoded classes:
0 -> negative
1 -> positive

Train size: 24914
Val size: 3114
Test size: 3115

Class weights:
{0: 1.2450208976921682, 1: 0.8956715559390279}


I0000 00:00:1776515676.792342      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1776515676.794679      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
260/260 ━━━━━━━━━━━━━━━━━━━━ 0s 736ms/step - accuracy: 0.4798 - loss: 0.7670
Epoch 1: val_loss improved from inf to 0.46878, saving model to /kaggle/working/tiktok_sentiment_binary/best_binary.keras
260/260 ━━━━━━━━━━━━━━━━━━━━ 215s 760ms/step - accuracy: 0.4800 - loss: 0.7669 - val_accuracy: 0.8060 - val_loss: 0.4688 - learning_rate: 3.0000e-04
Epoch 2/15
260/260 ━━━━━━━━━━━━━━━━━━━━ 0s 739ms/step - accuracy: 0.8128 - loss: 0.4660
Epoch 2: val_loss improved from 0.46878 to 0.35689, saving model to /kaggle/working/tiktok_sentiment_binary/best_binary.keras
260/260 ━━━━━━━━━━━━━━━━━━━━ 197s 757ms/step - accuracy: 0.8129 - loss: 0.4658 - val_accuracy: 0.8606 - val_loss: 0.3569 - learning_rate: 3.0000e-04
Epoch 3/15
260/260 ━━━━━━━━━━━━━━━━━━━━ 0s 740ms/step - accuracy: 0.8990 - loss: 0.2964
Epoch 3: val_loss improved from 0.35689 to 0.35472, saving model to /kaggle/working/tiktok_sentiment_binary/best_binary.keras
260/260 ━━━━━━━━━━━━━━━━━━━━ 197s 758ms/step - accuracy: 0.8990 

In [2]:
# =========================================
# SAVE FINAL MODEL ARTIFACTS
# =========================================
import os
import json
import pickle
import shutil

SAVE_DIR = OUTPUT_DIR
os.makedirs(SAVE_DIR, exist_ok=True)

# 1) Save final trained model
model.save(os.path.join(SAVE_DIR, "final_binary.keras"))

# 2) Copy best model with a clearer approved name
best_src = os.path.join(SAVE_DIR, "best_binary.keras")
best_dst = os.path.join(SAVE_DIR, "approved_best_binary.keras")

if os.path.exists(best_src):
    shutil.copy(best_src, best_dst)
    print("Best model copied to:", best_dst)
else:
    print("best_binary.keras not found")

# 3) Save tokenizer
with open(os.path.join(SAVE_DIR, "tokenizer.pkl"), "wb") as f:
    pickle.dump(tokenizer, f)

# 4) Save label encoder
with open(os.path.join(SAVE_DIR, "label_encoder.pkl"), "wb") as f:
    pickle.dump(le, f)

# 5) Save model info / config
save_info = {
    "TEXT_COL": TEXT_COL,
    "classes": list(le.classes_),
    "MAX_WORDS": MAX_WORDS,
    "MAX_LEN": MAX_LEN,
    "EMBED_DIM": EMBED_DIM,
    "BATCH_SIZE": BATCH_SIZE,
    "EPOCHS": EPOCHS,
    "LEARNING_RATE": LEARNING_RATE,
    "OUTPUT_DIR": SAVE_DIR
}

with open(os.path.join(SAVE_DIR, "saved_model_info.json"), "w", encoding="utf-8") as f:
    json.dump(save_info, f, ensure_ascii=False, indent=2)

print("\nSaved successfully:")
print(os.path.join(SAVE_DIR, "final_binary.keras"))
print(os.path.join(SAVE_DIR, "approved_best_binary.keras"))
print(os.path.join(SAVE_DIR, "tokenizer.pkl"))
print(os.path.join(SAVE_DIR, "label_encoder.pkl"))
print(os.path.join(SAVE_DIR, "saved_model_info.json"))

Best model copied to: /kaggle/working/tiktok_sentiment_binary/approved_best_binary.keras

Saved successfully:
/kaggle/working/tiktok_sentiment_binary/final_binary.keras
/kaggle/working/tiktok_sentiment_binary/approved_best_binary.keras
/kaggle/working/tiktok_sentiment_binary/tokenizer.pkl
/kaggle/working/tiktok_sentiment_binary/label_encoder.pkl
/kaggle/working/tiktok_sentiment_binary/saved_model_info.json
